# Funky Radical Electrodynamics Engine 1105 #

This Project (which can be abbreviated to FREE1105) Will Exist in 5 Phases:

1) Set Up the Electric Fields in a 2D Coordinate System
2) Simulate the Motion of Electric Point Charges
3) Simluate the Motion of Charged Rigid Bodies
4) Visualise the Field via Geometrical Curvature
5) Implement Maxwell's 4th Law for Magnetic Fields and Light

Bismillah Ar Rahman Ar Rahim - In Sha Allah this will go well!!

## Phase 1 ##

Let us establish the checklist of aims:
- Set Up the window to initialise a screen
- Add a wall on the screen so as to not have particles go so far off screen
- Be able to add particles on the 2d Screen to form point charges with initial positions
- Implement these point charges to be able to be static, environmental boolean values
- Implement the necassary equations for superpositions of electric fields
- Visualise the electric field in 2d when static as a heatmap in the baackground colour

Points of Refinement After
- implement a reset button to eliminate all non environmental charges and take everything back to inintial position and initial velocity
- implement a right click and drop down menu to then place components like point charges and then to drag them around

Alright so In Sha Allah lets do this and vaguely follow some vague classical approximation of physics in the process!!

#### Planning the Architecture of Phase 1 ####

We start with 5 files:
- the main pygame file
- constants imported for all files
- point charge class information
- electric field file
- gui file

Rough Plan:

1) The main pygame loop will load all the imports, initialise a screen and then set up a wall entity and throw it on there on the boundaries of the screen
2) write in the constants file that is imported for everything
2) Then in the point charge file, I will make a point charge class with attributes charge, mass, core_radius, border_radius, total_radius, position, pos_0, position, vel_0, velocity, environmental, static
3) Then create a function to relatively determine core_radius in comparison to other charges and a function to do the same for the border radius with mass. Then total radius is their sum
4) Create the electric field calculator and heatmap. Create a loop going through every pixel of a 1/4 scale virtual map and scale upwards to the full screen where in each pixel, the total net field at that space is calculated. 
5) heatmap is produced where effectively the background colour changes from default grey. at highest positive charge, it is blackest and at lowest negative charge, it is whitest. The Scale of charge will be symmetrical, so if the highest + charge is like idk 20 and highest negative is like 5, the 5 will be only 25% between grey and white whereas the posiitve will be fully black
6) make the gui file and make the reset button as a box in the bottom left as an object of its own class
7) make a drop down menu show up when you do a right click with for now just point charge. Then you click on the button and it opens a box containing options for its creation, being charge, mass, environmental and static toggles and the initial position is where the right click was
8) allow the point charge to get right clicked on, with options to drag to another position for now and allowing you to update the position.

#### How it Actually Went 1.0 ####

First off, I vibecoded the hell outta the GUI file. Pygame GUI isnt that tricky but oh my word, I genuinely couldn't care less forcing myself to make a class for toggles and text boxes. Otherwise it went good. I consulted Gemini on how to best implement the background screen so I could represent the electric field and it said to do a virtual screen and scale up 4x for 16x better performance. I tinkered with the colours a lot and yippee, I am happy with the end results.

Several times I vibe coded the GUI, lost complete track of what was going on and just like gave up and started again. I HATE GUI STUFF I SUCK. THE MATHS AND PHYSICS IS EASY. MICROMANAGING THE PIXEL DISTANCES SO IT DOES NOT LOOK LIKE GARBAGE SUCKS!! 

In [ ]:
%%writefile constants_for_all_files_jypter1.py

# below is constants_for_all_files.py

# necessary imports

import pygame
import numpy as np

# dimensions

SW = 1500 # Screen Width
SH = 900  # Screen Height
# (1/4 scale for E-Field calc) then scale up for perfomance
V_SW = SW // 4 # Virtual Width 
V_SH = SH // 4 # Virtual Height 

# default colors + wall stuff

BGC = 128, 128, 128  # Background Color for Neutral Grey
WC = (0,0, 0) # Neon Green for Wall Color
text_c = (30,30, 30) # Dark Grey for Text Color
WALL_RECT = pygame.Rect(0, 0, SW, SH)

# physics constants

WBT = 20 # Wall Border Thickness
BW_coeff = 0.9 # Coefficient of Restitution for Bouncing Wall
K_COULOMB = 8.99e9 # Coulomb's Constant in N m²/C²
E_CHARGE = 1.602e-19 # Elementary Charge in Coulombs
E_MASS = 9.109e-31 # Electron Mass in kg
P_MASS = 1.672e-27 # Proton Mass in kg

In [ ]:
%%writefile gui_jypter1.py

import pygame
import numpy as np
from constants_for_all_files import *

# --- UI CONFIGURATION ---
FONT_NAME = 'Consolas'
FONT_SIZE = 16
COL_BG = (240, 240, 240)
COL_BORDER = (0, 0, 0)
COL_ACTIVE = (50, 150, 255) # Blue highlight
COL_INACTIVE = (150, 150, 150)
COL_TEXT = (0, 0, 0)

# Standard UI Colors (ON/OFF)
COL_TRUE_STD = (50, 200, 50)    # Green
COL_FALSE_STD = (200, 50, 50)   # Red

# Physics Colors (Positive/Negative)
COL_POS = (200, 0, 0)   # Red
COL_NEG = (0, 0, 200)   # Blue

class ResetButton:
    def __init__(self):
        # Bottom Left Placement
        self.rect = pygame.Rect(20, SH - 60, 100, 40)
        self.font = pygame.font.SysFont('Arial', 20, bold=True)
        self.hovered = False

    def handle_event(self, event):
        if event.type == pygame.MOUSEMOTION:
            self.hovered = self.rect.collidepoint(event.pos)
        elif event.type == pygame.MOUSEBUTTONDOWN and event.button == 1:
            if self.hovered:
                return True
        return False

    def render(self, screen):
        color = (200, 50, 50) if self.hovered else (150, 50, 50)
        pygame.draw.rect(screen, color, self.rect)
        pygame.draw.rect(screen, (0,0,0), self.rect, 2)
        
        text = self.font.render("RESET", True, (255, 255, 255))
        text_rect = text.get_rect(center=self.rect.center)
        screen.blit(text, text_rect)

class StartButton:
    def __init__(self):
        # Placed next to ResetButton
        self.rect = pygame.Rect(1380, SH - 60, 100, 40)
        self.font = pygame.font.SysFont('Roboto', 24, bold=True)
        self.hovered = False
        
    def handle_event(self, event):
        if event.type == pygame.MOUSEMOTION:
            self.hovered = self.rect.collidepoint(event.pos)
        elif event.type == pygame.MOUSEBUTTONDOWN and event.button == 1:
            if self.hovered:
                return True
        return False

    def render(self, screen):
        # Green Color for Start/Run
        color = (50, 150, 50) if self.hovered else (50, 100, 50)
        pygame.draw.rect(screen, color, self.rect)
        pygame.draw.rect(screen, (0,0,0), self.rect, 2)
        
        text = self.font.render("START", True, (255, 255, 255))
        text_rect = text.get_rect(center=self.rect.center)
        screen.blit(text, text_rect)

class WallCORSlider:
    """Slider for wall coefficient of restitution (0.0 to 1.0)"""
    def __init__(self, x, y, width, height, initial_value=0.9):
        self.rect = pygame.Rect(x, y, width, height)
        self.slider_rect = pygame.Rect(x, y + height // 2 - 5, width, 10)
        self.knob_radius = 8
        self.value = initial_value
        self.dragging = False
        self.font = pygame.font.SysFont('Arial', 14)
        self.label = "Wall Bounce:"
        
    def handle_event(self, event):
        if event.type == pygame.MOUSEBUTTONDOWN and event.button == 1:
            knob_x = self.slider_rect.x + self.value * self.slider_rect.width
            knob_rect = pygame.Rect(knob_x - self.knob_radius, self.slider_rect.y - self.knob_radius, 
                                   self.knob_radius * 2, self.knob_radius * 2)
            if knob_rect.collidepoint(event.pos):
                self.dragging = True
                
        elif event.type == pygame.MOUSEBUTTONUP and event.button == 1:
            self.dragging = False
            
        elif event.type == pygame.MOUSEMOTION and self.dragging:
            # Update value based on mouse position
            relative_x = event.pos[0] - self.slider_rect.x
            self.value = max(0.0, min(1.0, relative_x / self.slider_rect.width))
            
    def render(self, screen):
        # Draw label
        label_surf = self.font.render(self.label, True, (0, 0, 0))
        screen.blit(label_surf, (self.rect.x, self.rect.y - 20))
        
        # Draw track
        pygame.draw.rect(screen, (200, 200, 200), self.slider_rect)
        pygame.draw.rect(screen, (0, 0, 0), self.slider_rect, 1)
        
        # Draw filled portion (progress bar)
        filled_width = self.slider_rect.width * self.value
        filled_rect = pygame.Rect(self.slider_rect.x, self.slider_rect.y, filled_width, self.slider_rect.height)
        pygame.draw.rect(screen, (100, 150, 255), filled_rect)
        
        # Draw knob
        knob_x = self.slider_rect.x + self.value * self.slider_rect.width
        knob_color = (50, 100, 200) if self.dragging else (100, 150, 255)
        pygame.draw.circle(screen, knob_color, (int(knob_x), int(self.slider_rect.centery)), self.knob_radius)
        pygame.draw.circle(screen, (0, 0, 0), (int(knob_x), int(self.slider_rect.centery)), self.knob_radius, 2)
        
        # Draw value text
        value_text = self.font.render(f"{self.value:.2f}", True, (0, 0, 0))
        screen.blit(value_text, (self.slider_rect.right + 10, self.slider_rect.centery - 7))

class ContextMenu:
    def __init__(self):
        self.active = False
        self.pos = (0, 0)
        self.rect = pygame.Rect(0, 0, 160, 30)
        self.font = pygame.font.SysFont('Arial', 16)

    def show(self, pos):
        self.active = True
        self.pos = pos
        self.rect.topleft = pos

    def handle_event(self, event):
        if not self.active: return None
        
        if event.type == pygame.MOUSEBUTTONDOWN and event.button == 1:
            if self.rect.collidepoint(event.pos):
                return "open_creator" # Clicked "Point Charge"
            else:
                self.active = False # Clicked away
        return None

    def render(self, screen):
        if not self.active: return
        pygame.draw.rect(screen, (240, 240, 240), self.rect)
        pygame.draw.rect(screen, (0,0,0), self.rect, 1)
        text = self.font.render("Create Charge", True, (0,0,0))
        screen.blit(text, (self.rect.x + 10, self.rect.y + 5))

# --- TEXT/TOGGLE COMPONENTS ---

class ScientificTextBox:
    """A text box that strictly accepts scientific notation inputs."""
    def __init__(self, x, y, w, h, default_text="1.0e-6"):
        self.rect = pygame.Rect(x, y, w, h)
        self.text = default_text
        self.active = False
        self.font = pygame.font.SysFont(FONT_NAME, FONT_SIZE)
        
    def handle_event(self, event):
        if event.type == pygame.MOUSEBUTTONDOWN:
            if self.rect.collidepoint(event.pos):
                self.active = True
            else:
                self.active = False
                
        if event.type == pygame.KEYDOWN and self.active:
            if event.key == pygame.K_RETURN:
                self.active = False
            elif event.key == pygame.K_BACKSPACE:
                self.text = self.text[:-1]
            else:
                # Strict filter: Digits, ., -, +, e, E
                if event.unicode in "0123456789.-+eE":
                    self.text += event.unicode

    def get_value(self):
        """Returns float if valid, None otherwise."""
        try:
            return float(self.text)
        except ValueError:
            return None

    def render(self, screen):
        # Border color indicates focus
        color = COL_ACTIVE if self.active else COL_INACTIVE
        pygame.draw.rect(screen, (255, 255, 255), self.rect)
        pygame.draw.rect(screen, color, self.rect, 2)
        
        # Text rendering
        text_surf = self.font.render(self.text, True, COL_TEXT)
        # Center text vertically
        text_rect = text_surf.get_rect(midleft=(self.rect.x + 5, self.rect.centery))
        screen.blit(text_surf, text_rect)

class MantissaTextBox:
    """Text box that enforces x.y format (max 2 digits before decimal)."""
    def __init__(self, x, y, w, h, default_text="1.0"):
        self.rect = pygame.Rect(x, y, w, h)
        self.text = default_text
        self.active = False
        self.font = pygame.font.SysFont(FONT_NAME, FONT_SIZE)
        
    def handle_event(self, event):
        if event.type == pygame.MOUSEBUTTONDOWN:
            if self.rect.collidepoint(event.pos):
                self.active = True
            else:
                self.active = False
                
        if event.type == pygame.KEYDOWN and self.active:
            if event.key == pygame.K_RETURN:
                self.active = False
            elif event.key == pygame.K_BACKSPACE:
                self.text = self.text[:-1]
            else:
                # Allow digits and one decimal point
                if event.unicode in "0123456789.":
                    # Prevent multiple decimal points
                    if event.unicode == "." and "." in self.text:
                        return
                    # Prevent more than 2 digits before decimal
                    if event.unicode.isdigit():
                        parts = self.text.split(".")
                        if len(parts[0]) >= 2:
                            return
                    self.text += event.unicode

    def get_value(self):
        """Returns float if valid, None otherwise."""
        try:
            val = float(self.text)
            return val if val > 0 else None
        except ValueError:
            return None

    def render(self, screen):
        color = COL_ACTIVE if self.active else COL_INACTIVE
        pygame.draw.rect(screen, (255, 255, 255), self.rect)
        pygame.draw.rect(screen, color, self.rect, 2)
        
        text_surf = self.font.render(self.text, True, COL_TEXT)
        text_rect = text_surf.get_rect(midleft=(self.rect.x + 5, self.rect.centery))
        screen.blit(text_surf, text_rect)

class ExponentTextBox:
    """Text box that enforces integer exponent from -99 to 99."""
    def __init__(self, x, y, w, h, default_text="0"):
        self.rect = pygame.Rect(x, y, w, h)
        self.text = default_text
        self.active = False
        self.font = pygame.font.SysFont(FONT_NAME, FONT_SIZE)
        
    def handle_event(self, event):
        if event.type == pygame.MOUSEBUTTONDOWN:
            if self.rect.collidepoint(event.pos):
                self.active = True
            else:
                self.active = False
                
        if event.type == pygame.KEYDOWN and self.active:
            if event.key == pygame.K_RETURN:
                self.active = False
            elif event.key == pygame.K_BACKSPACE:
                self.text = self.text[:-1]
            else:
                # Allow digits and one minus sign at the start
                if event.unicode in "0123456789-":
                    # Minus sign only at start and only one
                    if event.unicode == "-":
                        if len(self.text) == 0 or self.text[0] == "-":
                            if len(self.text) == 0:
                                self.text = "-"
                        return
                    # Limit to 2 digits (+ potential minus sign)
                    max_len = 3 if self.text.startswith("-") else 2
                    if len(self.text) < max_len:
                        self.text += event.unicode

    def get_value(self):
        """Returns float if valid, None otherwise."""
        try:
            val = int(self.text)
            return float(val) if -99 <= val <= 99 else None
        except ValueError:
            return None

    def render(self, screen):
        color = COL_ACTIVE if self.active else COL_INACTIVE
        pygame.draw.rect(screen, (255, 255, 255), self.rect)
        pygame.draw.rect(screen, color, self.rect, 2)
        
        text_surf = self.font.render(self.text, True, COL_TEXT)
        text_rect = text_surf.get_rect(midleft=(self.rect.x + 5, self.rect.centery))
        screen.blit(text_surf, text_rect)

class BinaryToggle:
    """A generic toggle button with customizable colors and text."""
    def __init__(self, x, y, w, h, label, initial_state=True, 
                 text_true="ON", text_false="OFF", 
                 col_true=COL_TRUE_STD, col_false=COL_FALSE_STD):
        
        self.rect = pygame.Rect(x, y, w, h)
        self.label = label
        self.state = initial_state
        self.font = pygame.font.SysFont('Arial', 14, bold=True)
        
        # Custom Configuration
        self.text_true = text_true
        self.text_false = text_false
        self.col_true = col_true
        self.col_false = col_false

    def handle_event(self, event):
        if event.type == pygame.MOUSEBUTTONDOWN and event.button == 1:
            if self.rect.collidepoint(event.pos):
                self.state = not self.state
                return True
        return False

    def render(self, screen):
        # Determine Color and Text based on state
        if self.state:
            bg_col = self.col_true
            status_txt = self.text_true
        else:
            bg_col = self.col_false
            status_txt = self.text_false
            
        pygame.draw.rect(screen, bg_col, self.rect)
        pygame.draw.rect(screen, (0,0,0), self.rect, 2)
        
        # Status Text inside button
        txt_surf = self.font.render(status_txt, True, (255,255,255))
        txt_rect = txt_surf.get_rect(center=self.rect.center)
        screen.blit(txt_surf, txt_rect)
        
        # Label Text above button
        if self.label:
            label_surf = self.font.render(self.label, True, (0,0,0))
            screen.blit(label_surf, (self.rect.x, self.rect.y - 18))

class CreationForm:
    def __init__(self):
        self.active = False
        self.rect = pygame.Rect(0, 0, 380, 360)
        self.target_pos = (0,0)
        self.font = pygame.font.SysFont('Arial', 16, bold=True)
        self.edit_mode = False  # False = Create, True = Edit
        self.editing_particle = None  # Reference to particle being edited
        
        # --- COMPONENTS ---
        
        # 1. Remember Toggle (Standard Green/Red)
        self.btn_remember = BinaryToggle(0, 0, 120, 25, "Remember Input", False)
        
        # 2. Charge Input & Sign Toggle (CUSTOM RED/BLUE)
        self.btn_charge_sign = BinaryToggle(0, 0, 50, 30, "", True, 
                                            text_true="+", text_false="-",
                                            col_true=COL_POS, col_false=COL_NEG)
        
        self.input_charge_mantissa = MantissaTextBox(0, 0, 80, 30, "1.0")
        self.input_charge_exponent = ExponentTextBox(0, 0, 60, 30, "0")
        
        # 3. Mass Input
        self.input_mass_mantissa = MantissaTextBox(0, 0, 80, 30, "1.0")
        self.input_mass_exponent = ExponentTextBox(0, 0, 60, 30, "0")
        
        # 4. Environmental & Static Toggles (Standard Green/Red)
        self.btn_env = BinaryToggle(0, 0, 60, 30, "Env", True)
        self.btn_static = BinaryToggle(0, 0, 60, 30, "Static", True)
        
        # 5. Action Buttons
        self.btn_submit = pygame.Rect(0,0, 100, 30)
        self.btn_cancel = pygame.Rect(0,0, 100, 30)

    def show(self, pos, particle=None):
        """Show form. If particle is provided, enter edit mode."""
        self.active = True
        self.target_pos = pos
        self.edit_mode = particle is not None
        self.editing_particle = particle
        self.rect.center = (SW//2, SH//2)
        
        if self.edit_mode and particle:
            # Load particle's current values into form
            # Decompose charge into mantissa and exponent
            charge_val = abs(particle.charge)
            if charge_val == 0:
                c_mantissa, c_exponent = 1.0, 0
            else:
                import math
                c_exponent = int(math.floor(math.log10(charge_val)))
                c_mantissa = charge_val / (10 ** c_exponent)
            
            # Decompose mass into mantissa and exponent
            mass_val = particle.mass
            if mass_val == 0:
                m_mantissa, m_exponent = 1.0, 0
            else:
                import math
                m_exponent = int(math.floor(math.log10(mass_val)))
                m_mantissa = mass_val / (10 ** m_exponent)
            
            self.input_charge_mantissa.text = f"{c_mantissa:.1f}"
            self.input_charge_exponent.text = str(c_exponent)
            self.input_mass_mantissa.text = f"{m_mantissa:.1f}"
            self.input_mass_exponent.text = str(m_exponent)
            self.btn_charge_sign.state = particle.charge >= 0
            self.btn_env.state = particle.environmental
            self.btn_static.state = particle.static
        else:
            # Create mode: reset to defaults
            self.input_charge_mantissa.text = "1.0"
            self.input_charge_exponent.text = "0"
            self.input_mass_mantissa.text = "1.0"
            self.input_mass_exponent.text = "0"
            self.btn_charge_sign.state = True
            self.btn_env.state = True
            self.btn_static.state = True
            
        # --- LAYOUT CALCULATION ---
        rx, ry = self.rect.x, self.rect.y
        
        # Remember Input (Top Right)
        self.btn_remember.rect.topright = (self.rect.right - 20, ry + 30)
        
        # Charge Row (Moved down to Y=90 for label clearance)
        self.btn_charge_sign.rect.topleft = (rx + 40, ry + 90)
        self.input_charge_mantissa.rect.topleft = (rx + 100, ry + 90)
        self.input_charge_exponent.rect.topleft = (rx + 230, ry + 90)
        
        # Mass Row (Moved down to Y=160)
        self.input_mass_mantissa.rect.topleft = (rx + 100, ry + 160)
        self.input_mass_exponent.rect.topleft = (rx + 230, ry + 160)
        
        # Toggles Row
        self.btn_env.rect.topleft = (rx + 50, ry + 230)
        self.btn_static.rect.topleft = (rx + 200, ry + 230)
        
        # Buttons
        self.btn_submit.bottomleft = (rx + 30, self.rect.bottom - 20)
        self.btn_cancel.bottomright = (self.rect.right - 30, self.rect.bottom - 20)

    def handle_event(self, event):
        if not self.active: return None

        # Pass events to children
        self.input_charge_mantissa.handle_event(event)
        self.input_charge_exponent.handle_event(event)
        self.input_mass_mantissa.handle_event(event)
        self.input_mass_exponent.handle_event(event)
        self.btn_remember.handle_event(event)
        self.btn_charge_sign.handle_event(event)
        self.btn_env.handle_event(event)
        self.btn_static.handle_event(event)

        if event.type == pygame.MOUSEBUTTONDOWN and event.button == 1:
            # Check Submit
            if self.btn_submit.collidepoint(event.pos):
                
                # Validate Inputs
                c_mantissa = self.input_charge_mantissa.get_value()
                c_exponent = self.input_charge_exponent.get_value()
                m_mantissa = self.input_mass_mantissa.get_value()
                m_exponent = self.input_mass_exponent.get_value()
                
                print(f"DEBUG: c_mantissa={c_mantissa}, c_exponent={c_exponent}, m_mantissa={m_mantissa}, m_exponent={m_exponent}")
                
                # Validation: All must be numbers, mantissas in range [1.0-9.9], mass > 0
                if (c_mantissa is not None and c_exponent is not None and 
                    m_mantissa is not None and m_exponent is not None and
                    1.0 <= c_mantissa <= 9.9 and 1.0 <= m_mantissa <= 9.9):
                    
                    # Construct charge value: mantissa * 10^exponent
                    charge = c_mantissa * (10 ** c_exponent)
                    if not self.btn_charge_sign.state: # If state is False (Blue/-)
                        charge = -charge
                    
                    # Construct mass value: mantissa * 10^exponent
                    mass = m_mantissa * (10 ** m_exponent)
                    
                    print(f"DEBUG: Validation passed. charge={charge}, mass={mass}")
                    
                    if mass > 0:
                        self.active = False
                        result = {
                            "charge": charge,
                            "mass": mass,
                            "environmental": self.btn_env.state,
                            "static": self.btn_static.state
                        }
                        
                        if self.edit_mode:
                            # Edit mode: return particle reference and edited flag
                            result["particle"] = self.editing_particle
                            result["edit"] = True
                        else:
                            # Create mode: return position for new particle
                            result["position"] = np.array(self.target_pos, dtype=float)
                        
                        return result
            
            # Check Cancel
            elif self.btn_cancel.collidepoint(event.pos):
                self.active = False
                return None
                
        return None

    def render(self, screen):
        if not self.active: return
        
        # Draw Panel
        pygame.draw.rect(screen, COL_BG, self.rect)
        pygame.draw.rect(screen, COL_BORDER, self.rect, 2)
        
        # Title (different based on mode)
        title_text = "Edit Charge" if self.edit_mode else "Create Charge"
        title = self.font.render(title_text, True, COL_TEXT)
        screen.blit(title, (self.rect.x + 20, self.rect.y + 20))
        
        # Labels for TextBoxes
        l_chg = self.font.render("Charge:", True, COL_TEXT)
        screen.blit(l_chg, (self.rect.x + 100, self.input_charge_mantissa.rect.top - 20))
        
        l_mass = self.font.render("Mass:", True, COL_TEXT)
        screen.blit(l_mass, (self.rect.x + 100, self.input_mass_mantissa.rect.top - 20))
        
        # x 10^ labels
        exp_label_c = self.font.render("x 10 ^", True, COL_TEXT)
        screen.blit(exp_label_c, (self.input_charge_mantissa.rect.right + 10, self.input_charge_mantissa.rect.centery - 8))
        
        exp_label_m = self.font.render("x 10 ^", True, COL_TEXT)
        screen.blit(exp_label_m, (self.input_mass_mantissa.rect.right + 10, self.input_mass_mantissa.rect.centery - 8))
        
        # Render Components
        self.btn_remember.render(screen)
        self.btn_charge_sign.render(screen)
        self.input_charge_mantissa.render(screen)
        self.input_charge_exponent.render(screen)
        self.input_mass_mantissa.render(screen)
        self.input_mass_exponent.render(screen)
        self.btn_env.render(screen)
        self.btn_static.render(screen)
        
        # Render Action Buttons
        # Submit (Green-ish)
        pygame.draw.rect(screen, (100, 200, 100), self.btn_submit)
        pygame.draw.rect(screen, (0,0,0), self.btn_submit, 1)
        submit_text = "CHANGE" if self.edit_mode else "SPAWN"
        ts = self.font.render(submit_text, True, (0,0,0))
        ts_rect = ts.get_rect(center=self.btn_submit.center)
        screen.blit(ts, ts_rect)
        
        # Cancel (Red-ish)
        pygame.draw.rect(screen, (200, 100, 100), self.btn_cancel)
        pygame.draw.rect(screen, (0,0,0), self.btn_cancel, 1)
        tc = self.font.render("CANCEL", True, (0,0,0))
        tc_rect = tc.get_rect(center=self.btn_cancel.center)
        screen.blit(tc, tc_rect)

In [ ]:
%%writefile point_charges_jypter1.py

# below is point_charges.py

# necessary imports

import pygame 
import numpy as np
from constants_for_all_files import *

# Point Charge Class

class PointCharge:

    def __init__(self, pos_0, charge, mass, environmental, static, pc_id):
        self.position = np.array(pos_0, dtype=float) # Position Vector
        self.vel = np.array([0.0, 0.0], dtype=float) # Velocity Vector
        self.pos_0 = np.array(pos_0, dtype=float) # Initial Position Vector
        self.vel_0 = np.array([0.0, 0.0], dtype=float) # Initial Velocity Vector
        self.charge = charge # Charge in Coulombs
        self.mass = mass # Mass in kg
        self.environmental = environmental # Boolean for Environmental Charge
        self.static = static # Boolean for Static Charge
        self.pc_id = pc_id # Unique ID for Point Charge
        self.dragging = False # Boolean for Dragging State

        # Initialize radius values (will be updated by update_relative_scale)
        self.core_radius = 15  # default core radius
        self.border_radius = 6  # default border radius
        self.total_radius = self.core_radius + self.border_radius

        self.color = (200, 0, 0) if charge > 0 else (0, 0, 200) # Red for Positive, Blue for Negative

    @property
    def pos(self):
        """Property to always return current position (for compatibility with electric_field.py)."""
        return self.position

    def update_relative_scale(self, all_charges):
        
        # Determines core_radius based on charge magnitude vs others.
        # Determines border_radius based on mass vs others.
        
        if not all_charges:
            return

        # 1. Calculate Core Radius (Charge)
        max_q = max(abs(pc.charge) for pc in all_charges)
        min_q = min(abs(pc.charge) for pc in all_charges)
        q_range = max_q - min_q
        
        current_q = abs(self.charge)
        
        if q_range == 0:
            norm_q = 0.5
        else:
            # (current - min) / range -> 0 to 1
            norm_q = (current_q - min_q) / q_range
            
        self.core_radius = 5 + (norm_q * 15) # Scale 5px to 20px

        # 2. Calculate Border Radius (Mass)
        max_m = max(pc.mass for pc in all_charges)
        min_m = min(pc.mass for pc in all_charges)
        m_range = max_m - min_m
        
        current_m = self.mass
        
        if m_range == 0:
            norm_m = 0.5
        else:
            norm_m = (current_m - min_m) / m_range
            
        self.border_radius = 1 + (norm_m * 4) # Scale 1px to 5px
        
        # 3. Sum for total hitbox
        self.total_radius = self.core_radius + self.border_radius

    def reset(self):

        # reverts to initial state
        
        if not self.environmental:
            self.pos = self.pos_0.copy()
            self.vel = self.vel_0.copy()
        else:
            # delete the point charge if NOT environmental
            del self
    
    def render(self, screen):
        # Draw Border (Mass representation) - Colored based on charge
        if self.charge > 0:
            border_colour_fill = (220, 90, 90)  # Orange for Positive Charge Border
        elif self.charge < 0:
            border_colour_fill = (130, 20, 130)  # Purple for Negative Charge Border
        else:
            border_colour_fill = (100, 100, 100)  # Grey for neutral
            
        pygame.draw.circle(screen, border_colour_fill, self.position.astype(int), int(self.total_radius))
        
        # Draw Core (Charge representation) - Colored
        pygame.draw.circle(screen, self.color, self.position.astype(int), int(self.core_radius))
        
        # Selection Highlight
        if self.dragging:
            pygame.draw.circle(screen, (255, 255, 255), self.position.astype(int), int(self.total_radius + 2), 2)
            # sets new position while dragging
            self.position = np.array(pygame.mouse.get_pos(), dtype=float)

In [ ]:
%%writefile electric_field_jypter1.py

# below is electric_field.py file

# necessary imports

import pygame
import numpy as np
from constants_for_all_files import *

# Electric Field Calculation and Visualization

class ElectricFieldSystem:

    def __init__(self):

        # sets up the virtual grid

        self.x_coords = np.linspace(0, SW, V_SW)
        self.y_coords = np.linspace(0, SH, V_SH)
        self.grid_x, self.grid_y = np.meshgrid(self.x_coords, self.y_coords)
        
        # surface to hold the heatmap

        self.surface = pygame.Surface((V_SW, V_SH))

    def render(self, target_surface, charges):

        if not charges:
            target_surface.fill(BGC)
            return

        # Calculate the electric field at each grid point due to all charges

        net_field = np.zeros_like(self.grid_x, dtype=float)

        for pc in charges:
            dx = self.grid_x - pc.pos[0]
            dy = self.grid_y - pc.pos[1]
            r2 = dx**2 + dy**2
            r2[r2 == 0] = E_CHARGE # Prevent division by zero
            min_distance = pc.total_radius ** 2
            r2[r2 < min_distance] = min_distance # Cap minimum distance squared to avoid extreme values

            contribution = (K_COULOMB * pc.charge) / r2
            net_field += contribution

        # colourises the heatmap based on field strength

        max_val = np.max(np.abs(net_field))
        if max_val == 0:
            max_val = 100  # so eveything is neutral grey

        normalised = 0.7 * net_field / max_val
        pixel_vals = 127.5 - (normalised * 127.5)
        pixel_vals = np.clip(pixel_vals, 0, 255).astype(np.uint8)
        
        # create RGB array for Pygame surface

        pixels = np.dstack([pixel_vals] * 3)
        pygame.surfarray.blit_array(self.surface, np.swapaxes(pixels, 0, 1))
        
        # scale up to full screen

        scaled_view = pygame.transform.scale(self.surface, (SW, SH))
        target_surface.blit(scaled_view, (0, 0))
            

In [ ]:
%%writefile main_jypter1.py

# below is the main.py file

# necessary imports

import pygame
import numpy as np
import sys

# import all other necessary modules

from constants_for_all_files_jypter1 import *
from point_charges_jypter1 import *
from electric_field_jypter1 import *
from gui_jypter1 import *

# set up the window

pygame.init()
screen = pygame.display.set_mode((SW, SH))
pygame.display.set_caption("FREE1105: Phase 1")
clock = pygame.time.Clock()

# setting up systems

ef_system = ElectricFieldSystem()
reset_btn = ResetButton()
context_menu = ContextMenu()
create_form = CreationForm()
start_btn = StartButton()
wall_cor_slider = WallCORSlider(30, 20, 150, 30, initial_value=BW_coeff)

# setting up initial state of the simulation

all_point_charges = []
selected_charge = None
p_id_counter = 0
wall_cor = wall_cor_slider.value  # Current wall coefficient of restitution

# the main loop where everything happens

running = True
while running:

    # event handling

    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False
            pygame.quit()
            sys.exit()

        # handle reset button

        if reset_btn.handle_event(event):
            all_point_charges = [pc for pc in all_point_charges if pc.environmental]
            for pc in all_point_charges:
                pc.reset()

        # handle start button

        if start_btn.handle_event(event):
            pass  # TODO: Start simulation logic

        # handle wall COR slider

        wall_cor_slider.handle_event(event)

        # handle context menu

        menu_action = context_menu.handle_event(event)
        if menu_action == "open_creator":
            create_form.show(context_menu.pos)

        # creation/edit form handling

        form_data = create_form.handle_event(event)
        if form_data:
            try:
                if form_data.get('edit'):
                    # Edit mode: update existing particle
                    particle = form_data['particle']
                    particle.charge = form_data['charge']
                    particle.mass = form_data['mass']
                    particle.environmental = form_data['environmental']
                    particle.static = form_data['static']
                    # Update color based on new charge sign
                    particle.color = (200, 0, 0) if particle.charge > 0 else (0, 0, 200)
                    particle.update_relative_scale(all_point_charges)
                    print(f"EDIT: Particle updated - charge={particle.charge}, mass={particle.mass}, env={particle.environmental}, static={particle.static}")
                else:
                    # Create mode: spawn new particle
                    new_pc = PointCharge(
                        pos_0=form_data['position'],
                        charge=form_data['charge'],
                        mass=form_data['mass'],
                        environmental=form_data['environmental'],
                        static=form_data['static'],
                        pc_id=p_id_counter
                    )
                    all_point_charges.append(new_pc)
                    p_id_counter += 1
            except Exception as e:
                import traceback
                print("Error in form submission:")
                print("form_data=", form_data)
                traceback.print_exc()

        # mouse interactions
        
        if event.type == pygame.MOUSEBUTTONDOWN:
            if event.button == 1: # Left Click (Drag)
                if not context_menu.active and not create_form.active:
                    for pc in reversed(all_point_charges):
                        # Check distance for click
                        dist = np.linalg.norm(pc.position - np.array(event.pos))
                        if dist < pc.total_radius:
                            selected_charge = pc
                            pc.dragging = True
                            break
                            
            elif event.button == 3: # Right Click (Menu or Edit)
                clicked_on_charge = False
                clicked_charge = None
                for pc in reversed(all_point_charges):
                     dist = np.linalg.norm(pc.position - np.array(event.pos))
                     if dist < pc.total_radius:
                         # Right clicked a charge - open edit form
                         clicked_on_charge = True
                         clicked_charge = pc
                         break
                
                if clicked_on_charge and clicked_charge:
                    # Open edit form for this charge
                    create_form.show(event.pos, particle=clicked_charge)
                else:
                    # Open context menu to create new charge
                    context_menu.show(event.pos)

        elif event.type == pygame.MOUSEBUTTONUP:
            if event.button == 1 and selected_charge:
                selected_charge.dragging = False
                selected_charge = None
                
        elif event.type == pygame.MOUSEMOTION:
            if selected_charge:
                selected_charge.position = np.array(event.pos, dtype=float)

    # logic and updates

    wall_cor = wall_cor_slider.value  # Update wall coefficient from slider

    for pc in all_point_charges:
        pc.update_relative_scale(all_point_charges)

    # rendering

    ef_system.render(screen, all_point_charges)
    pygame.draw.rect(screen, WC, WALL_RECT, WBT) # draw wall
    for pc in all_point_charges:
        pc.render(screen)

    # gui rendering

    reset_btn.render(screen)
    start_btn.render(screen)
    wall_cor_slider.render(screen)
    context_menu.render(screen)
    create_form.render(screen)

    pygame.display.flip()
    clock.tick(60)  # Limit to 60 FPS

pygame.quit()
sys.exit()

In [ ]:
from main_jypter1 import *

## Phase 2 ##

Let us establish the checklist of aims:
- When the Start button is pressed, stop allowing user interaction, unless reset, with only a pause button allowed
- Implement motion of the point charges from completely static spawn and when the reset button is clicked, the sim is stopped and they all go back to their initial positions or get OBLITERATED INTO OBLIVION if not self.environmental. And then you can drag and drop stuff for the sim
- Establish after effects and then after trails like in angry birds that disappear in like 10-30s idk
- Establish collision mechanics between particles and the wall with its sliding coefficient meter. I MAY implement variable coefficient of restitutions as an ability to create and change point charges
- Then finally implement some form of variable initial velocity. Exact details idk. Then implement an arrow indicator on the object before the simulation is run to indicate the direction and relative size. May make the arrow draggable to establish direction with more precision once in preview mode. But when the sim is running, its gone

Points for Refinement After:
- Time intervals and playback times will have to undergo significant scaling for the purpose of usability and visuals. 
- May experiment regarding how to visually represent the coefficient of restitution for things

#### Planning the Architecture of Phase 2 ####

We start with 5 files from Phase 1, All will likely be augumented:
- the main pygame file
- constants imported for all files
- point charge class information
- electric field file
- gui file

The Following new Files will be made:
- physics engine file

Rough Plan:

1) Implement the Start Button and Reset button functionality. A Finite State Machine will be implemented where the simulation is in one of 3 states - set up / running / pause. This will then affect the simulation loop accordingly. Also Do not run the simulation if any of the charges start out touching one another
2) Then implement the physics engine and motion in accordance to static starting objects. This will be done by via the Runge-Kutta Methods
3) Handle Collisions via for now fully elastic collisions
4) The Reset Button and Pause functionality will be bullet proofed here
5) Establish motion effects and aftertrails
6) Establish an ability to edit point charges and their coefficients of restitution like with the wall. Also when editting a point charge to be able to delete one.
7) Implement initial motion setting as something in the code
8) Make the motion of bearable viewing, ie time differences and etc. Gonna implement a slider to change the time step from 1/100 to 1/10000 or maybe higher, and a separate to change the clock speed from 60 fps to 900
9) Establish the GUI for this variable motion setting and have before the sim starts, the point charges show which direction the charge will go via some form of arrow with relative scaling that disappears once the simulation is started
10) some guy came by and told me to switch from rk4 to simpletic integrator because it respects like conservation of energy or something

#### How it Actually Went 2.0 ####

Oh my word so much vibecoding. This was just in general quite exhausting. I continued to shameleslly vibecode the GUI, and vibe coded the new physics engine. However, the rest was done Alhamdu'Lillah. Like I did code most of it, but so much getting something then debugging and then relearning my code and re understanding it. I spent a while then fine tuning the physics to vaguely at least follow energy saving laws. I'm now just overwhelmed with pushing further.

In [ ]:
%%writefile constants_for_all_files_jupyter2.py

# below is constants_for_all_files.py

# necessary imports

import pygame
import numpy as np

# dimensions

SW = 1500 # Screen Width
SH = 900  # Screen Height
# (1/4 scale for E-Field calc) then scale up for perfomance
V_SW = SW // 4 # Virtual Width 
V_SH = SH // 4 # Virtual Height 

# default colors + wall stuff

BGC = 128, 128, 128  # Background Color for Neutral Grey
WC = (0,0, 0) # Neon Green for Wall Color
text_c = (30,30, 30) # Dark Grey for Text Color
WALL_RECT = pygame.Rect(0, 0, SW, SH)

# physics constants

WBT = 20 # Wall Border Thickness
BW_coeff = 0.9 # Coefficient of Restitution for Bouncing Wall
# Inner wall rectangle (the visible wall border area is drawn using WALL_RECT and WBT).
# Use this inset rect as the actual collision boundary so particles bounce off the black wall
# rather than the outer window edges.
WALL_INNER_RECT = pygame.Rect(WBT, WBT, SW - 2 * WBT, SH - 2 * WBT)
K_COULOMB = 8.99e9 # Coulomb's Constant in N m²/C²
E_CHARGE = 1.602e-19 # Elementary Charge in Coulombs
E_MASS = 9.109e-31 # Electron Mass in kg
P_MASS = 1.672e-27 # Proton Mass in kg

# simulation states

sim_state = 0 # 0 = Setup, 1 = Running, 0.5 = Paused

# after effects and trails

TRAIL_LENGTH = 40   # How many "ghosts" to keep
TRAIL_SKIP = 2      # Record position every N frames (Optimization)

In [ ]:
%%writefile gui_jupyter2.py

# below is gui.py

import pygame
import numpy as np
from constants_for_all_files import *

# --- UI CONFIGURATION ---
FONT_NAME = 'Consolas'
FONT_SIZE = 16
COL_BG = (240, 240, 240)
COL_BORDER = (0, 0, 0)
COL_ACTIVE = (50, 150, 255) # Blue highlight
COL_INACTIVE = (150, 150, 150)
COL_TEXT = (0, 0, 0)

# Standard UI Colors (ON/OFF)
COL_TRUE_STD = (50, 200, 50)    # Green
COL_FALSE_STD = (200, 50, 50)   # Red

# Physics Colors (Positive/Negative)
COL_POS = (200, 0, 0)   # Red
COL_NEG = (0, 0, 200)   # Blue

class ResetButton:
    def __init__(self):
        # Bottom Left Placement
        self.rect = pygame.Rect(20, SH - 60, 100, 40)
        self.font = pygame.font.SysFont('Arial', 20, bold=True)
        self.hovered = False

    def handle_event(self, event):
        if event.type == pygame.MOUSEMOTION:
            self.hovered = self.rect.collidepoint(event.pos)
        elif event.type == pygame.MOUSEBUTTONDOWN and event.button == 1:
            if self.hovered:
                return True
        return False

    def render(self, screen):
        color = (200, 50, 50) if self.hovered else (150, 50, 50)
        pygame.draw.rect(screen, color, self.rect)
        pygame.draw.rect(screen, (0,0,0), self.rect, 2)
        
        text = self.font.render("RESET", True, (255, 255, 255))
        text_rect = text.get_rect(center=self.rect.center)
        screen.blit(text, text_rect)

class StartButton:
    def __init__(self):
        # Placed next to ResetButton
        self.rect = pygame.Rect(1380, SH - 60, 100, 40)
        self.font = pygame.font.SysFont('Roboto', 24, bold=True)
        self.hovered = False
        
    def handle_event(self, event):
        if event.type == pygame.MOUSEMOTION:
            self.hovered = self.rect.collidepoint(event.pos)
        elif event.type == pygame.MOUSEBUTTONDOWN and event.button == 1:
            if self.hovered:
                return True
        return False

    def render(self, screen):
        # Green Color for Start/Run
        color = (50, 150, 50) if self.hovered else (50, 100, 50)
        pygame.draw.rect(screen, color, self.rect)
        pygame.draw.rect(screen, (0,0,0), self.rect, 2)
        
        text = self.font.render("START", True, (255, 255, 255))
        text_rect = text.get_rect(center=self.rect.center)
        screen.blit(text, text_rect)

class PauseButton: 

    def __init__(self):
        # Placed instead of StartButton
        self.rect = pygame.Rect(1380, SH - 60, 100, 40)
        self.font = pygame.font.SysFont('Roboto', 24, bold=True)
        self.hovered = False
        
    def handle_event(self, event):
        if event.type == pygame.MOUSEMOTION:
            self.hovered = self.rect.collidepoint(event.pos)
        elif event.type == pygame.MOUSEBUTTONDOWN and event.button == 1:
            if self.hovered:
                return True
        return False

    def render(self, screen):
        # Yellow Color for Pause
        color = (200, 200, 50) if self.hovered else (150, 150, 50)
        pygame.draw.rect(screen, color, self.rect)
        pygame.draw.rect(screen, (0,0,0), self.rect, 2)
        
        text = self.font.render("PAUSE", True, (255, 255, 255))
        text_rect = text.get_rect(center=self.rect.center)
        screen.blit(text, text_rect)

class UnpauseButton(PauseButton):

    def render(self, screen):
        # Green Color for Unpause
        color = (50, 150, 50) if self.hovered else (50, 100, 50)
        pygame.draw.rect(screen, color, self.rect)
        pygame.draw.rect(screen, (0,0,0), self.rect, 2)
        self.rect = pygame.Rect(1380, SH - 60, 100, 40)
        text = self.font.render("UNPAUSE", True, (255, 255, 255))
        text_rect = text.get_rect(center=self.rect.center)
        screen.blit(text, text_rect)

class WallCORSlider:
    """Slider for wall coefficient of restitution (0.0 to 1.0)"""
    def __init__(self, x, y, width, height, initial_value=0.9):
        self.rect = pygame.Rect(x, y, width, height)
        self.slider_rect = pygame.Rect(x, y + height // 2 - 5, width, 10)
        self.knob_radius = 8
        self.value = initial_value
        self.dragging = False
        self.font = pygame.font.SysFont('Arial', 14)
        self.label = "Wall Bounce:"
        
    def handle_event(self, event):
        if event.type == pygame.MOUSEBUTTONDOWN and event.button == 1:
            knob_x = self.slider_rect.x + self.value * self.slider_rect.width
            knob_rect = pygame.Rect(knob_x - self.knob_radius, self.slider_rect.y - self.knob_radius, 
                                   self.knob_radius * 2, self.knob_radius * 2)
            if knob_rect.collidepoint(event.pos):
                self.dragging = True
                
        elif event.type == pygame.MOUSEBUTTONUP and event.button == 1:
            self.dragging = False
            
        elif event.type == pygame.MOUSEMOTION and self.dragging:
            # Update value based on mouse position
            relative_x = event.pos[0] - self.slider_rect.x
            self.value = max(0.0, min(1.0, relative_x / self.slider_rect.width))
            
    def render(self, screen):
        # Draw label
        #label_surf = self.font.render(self.label, True, (0, 0, 0))
        #screen.blit(label_surf, (self.rect.x, self.rect.y - 20))
        
        # Draw track
        pygame.draw.rect(screen, (200, 200, 200), self.slider_rect)
        pygame.draw.rect(screen, (0, 0, 0), self.slider_rect, 1)
        
        # Draw filled portion (progress bar)
        filled_width = self.slider_rect.width * self.value
        filled_rect = pygame.Rect(self.slider_rect.x, self.slider_rect.y, filled_width, self.slider_rect.height)
        pygame.draw.rect(screen, (100, 150, 255), filled_rect)
        
        # Draw knob
        knob_x = self.slider_rect.x + self.value * self.slider_rect.width
        knob_color = (50, 100, 200) if self.dragging else (100, 150, 255)
        pygame.draw.circle(screen, knob_color, (int(knob_x), int(self.slider_rect.centery)), self.knob_radius)
        pygame.draw.circle(screen, (0, 0, 0), (int(knob_x), int(self.slider_rect.centery)), self.knob_radius, 2)
        
        # Draw value text
        value_text = self.font.render(f"{self.value:.2f}", True, (0, 0, 0))
        screen.blit(value_text, (self.slider_rect.right + 10, self.slider_rect.centery - 7))

class ContextMenu:
    def __init__(self):
        self.active = False
        self.pos = (0, 0)
        self.rect = pygame.Rect(0, 0, 160, 30)
        self.font = pygame.font.SysFont('Arial', 16)

    def show(self, pos):
        self.active = True
        self.pos = pos
        self.rect.topleft = pos

    def handle_event(self, event):
        if not self.active: return None
        
        if event.type == pygame.MOUSEBUTTONDOWN and event.button == 1:
            if self.rect.collidepoint(event.pos):
                return "open_creator" # Clicked "Point Charge"
            else:
                self.active = False # Clicked away
        return None

    def render(self, screen):
        if not self.active: return
        pygame.draw.rect(screen, (240, 240, 240), self.rect)
        pygame.draw.rect(screen, (0,0,0), self.rect, 1)
        text = self.font.render("Create Charge", True, (0,0,0))
        screen.blit(text, (self.rect.x + 10, self.rect.y + 5))

# --- TEXT/TOGGLE COMPONENTS ---

class ScientificTextBox:
    """A text box that strictly accepts scientific notation inputs."""
    def __init__(self, x, y, w, h, default_text="1.0e-6"):
        self.rect = pygame.Rect(x, y, w, h)
        self.text = default_text
        self.active = False
        self.font = pygame.font.SysFont(FONT_NAME, FONT_SIZE)
        
    def handle_event(self, event):
        if event.type == pygame.MOUSEBUTTONDOWN:
            if self.rect.collidepoint(event.pos):
                self.active = True
            else:
                self.active = False
                
        if event.type == pygame.KEYDOWN and self.active:
            if event.key == pygame.K_RETURN:
                self.active = False
            elif event.key == pygame.K_BACKSPACE:
                self.text = self.text[:-1]
            else:
                # Strict filter: Digits, ., -, +, e, E
                if event.unicode in "0123456789.-+eE":
                    self.text += event.unicode

    def get_value(self):
        """Returns float if valid, None otherwise."""
        try:
            return float(self.text)
        except ValueError:
            return None

    def render(self, screen):
        # Border color indicates focus
        color = COL_ACTIVE if self.active else COL_INACTIVE
        pygame.draw.rect(screen, (255, 255, 255), self.rect)
        pygame.draw.rect(screen, color, self.rect, 2)
        
        # Text rendering
        text_surf = self.font.render(self.text, True, COL_TEXT)
        # Center text vertically
        text_rect = text_surf.get_rect(midleft=(self.rect.x + 5, self.rect.centery))
        screen.blit(text_surf, text_rect)

class MantissaTextBox:
    """Text box that enforces x.y format (max 2 digits before decimal)."""
    def __init__(self, x, y, w, h, default_text="1.0"):
        self.rect = pygame.Rect(x, y, w, h)
        self.text = default_text
        self.active = False
        self.font = pygame.font.SysFont(FONT_NAME, FONT_SIZE)
        self.allow0 = False
        
    def handle_event(self, event):
        if event.type == pygame.MOUSEBUTTONDOWN:
            if self.rect.collidepoint(event.pos):
                self.active = True
            else:
                self.active = False
                
        if event.type == pygame.KEYDOWN and self.active:
            if event.key == pygame.K_RETURN:
                self.active = False
            elif event.key == pygame.K_BACKSPACE:
                self.text = self.text[:-1]
            else:
                # Allow digits and one decimal point
                if event.unicode in "0123456789.":
                    # Prevent multiple decimal points
                    if event.unicode == "." and "." in self.text:
                        return
                    # Prevent more than 2 digits before decimal
                    if event.unicode.isdigit():
                        parts = self.text.split(".")
                        if len(parts[0]) >= 2:
                            return
                    self.text += event.unicode

    def get_value(self):
        """Returns float if valid, None otherwise."""
        try:
            val = float(self.text)
            if self.allow0:
                return val if val >= 0 else None
            else:
                return val if val > 0 else None
        except ValueError:
            return None

    def render(self, screen):
        color = COL_ACTIVE if self.active else COL_INACTIVE
        pygame.draw.rect(screen, (255, 255, 255), self.rect)
        pygame.draw.rect(screen, color, self.rect, 2)
        
        text_surf = self.font.render(self.text, True, COL_TEXT)
        text_rect = text_surf.get_rect(midleft=(self.rect.x + 5, self.rect.centery))
        screen.blit(text_surf, text_rect)

class ExponentTextBox:
    """Text box that enforces integer exponent from -99 to 99."""
    def __init__(self, x, y, w, h, default_text="0"):
        self.rect = pygame.Rect(x, y, w, h)
        self.text = default_text
        self.active = False
        self.font = pygame.font.SysFont(FONT_NAME, FONT_SIZE)
        
    def handle_event(self, event):
        if event.type == pygame.MOUSEBUTTONDOWN:
            if self.rect.collidepoint(event.pos):
                self.active = True
            else:
                self.active = False
                
        if event.type == pygame.KEYDOWN and self.active:
            if event.key == pygame.K_RETURN:
                self.active = False
            elif event.key == pygame.K_BACKSPACE:
                self.text = self.text[:-1]
            else:
                # Allow digits and one minus sign at the start
                if event.unicode in "0123456789-":
                    # Minus sign only at start and only one
                    if event.unicode == "-":
                        if len(self.text) == 0 or self.text[0] == "-":
                            if len(self.text) == 0:
                                self.text = "-"
                        return
                    # Limit to 2 digits (+ potential minus sign)
                    max_len = 3 if self.text.startswith("-") else 2
                    if len(self.text) < max_len:
                        self.text += event.unicode

    def get_value(self):
        """Returns float if valid, None otherwise."""
        try:
            val = int(self.text)
            return float(val) if -99 <= val <= 99 else None
        except ValueError:
            return None

    def render(self, screen):
        color = COL_ACTIVE if self.active else COL_INACTIVE
        pygame.draw.rect(screen, (255, 255, 255), self.rect)
        pygame.draw.rect(screen, color, self.rect, 2)
        
        text_surf = self.font.render(self.text, True, COL_TEXT)
        text_rect = text_surf.get_rect(midleft=(self.rect.x + 5, self.rect.centery))
        screen.blit(text_surf, text_rect)

class BinaryToggle:
    """A generic toggle button with customizable colors and text."""
    def __init__(self, x, y, w, h, label, initial_state=True, 
                 text_true="ON", text_false="OFF", 
                 col_true=COL_TRUE_STD, col_false=COL_FALSE_STD):
        
        self.rect = pygame.Rect(x, y, w, h)
        self.label = label
        self.state = initial_state
        self.font = pygame.font.SysFont('Arial', 14, bold=True)
        
        # Custom Configuration
        self.text_true = text_true
        self.text_false = text_false
        self.col_true = col_true
        self.col_false = col_false

    def handle_event(self, event):
        if event.type == pygame.MOUSEBUTTONDOWN and event.button == 1:
            if self.rect.collidepoint(event.pos):
                self.state = not self.state
                return True
        return False

    def render(self, screen):
        # Determine Color and Text based on state
        if self.state:
            bg_col = self.col_true
            status_txt = self.text_true
        else:
            bg_col = self.col_false
            status_txt = self.text_false
            
        pygame.draw.rect(screen, bg_col, self.rect)
        pygame.draw.rect(screen, (0,0,0), self.rect, 2)
        
        # Status Text inside button
        txt_surf = self.font.render(status_txt, True, (255,255,255))
        txt_rect = txt_surf.get_rect(center=self.rect.center)
        screen.blit(txt_surf, txt_rect)
        
        # Label Text above button
        if self.label:
            label_surf = self.font.render(self.label, True, (0,0,0))
            screen.blit(label_surf, (self.rect.x, self.rect.y - 18))

class TrailsToggle:
    """A simple toggle button labeled 'TRAILS' with white text and red/green background."""
    def __init__(self, x=140, y=SH-60, w=100, h=40, initial_state=False):
        self.rect = pygame.Rect(x, y, w, h)
        self.state = initial_state
        self.font = pygame.font.SysFont('Arial', 18, bold=True)
        self.hovered = False

    def handle_event(self, event):
        if event.type == pygame.MOUSEMOTION:
            self.hovered = self.rect.collidepoint(event.pos)
        if event.type == pygame.MOUSEBUTTONDOWN and event.button == 1:
            if self.rect.collidepoint(event.pos):
                self.state = not self.state
                return True
        return False

    def render(self, screen):
        # Background: green when ON, red when OFF
        bg = (50, 200, 50) if self.state else (200, 50, 50)
        # Slight brighten on hover
        if self.hovered:
            bg = tuple(min(255, int(c * 1.05)) for c in bg)
        pygame.draw.rect(screen, bg, self.rect)
        pygame.draw.rect(screen, (0,0,0), self.rect, 2)

        # White label centered
        txt = self.font.render("TRAILS", True, (255, 255, 255))
        txt_rect = txt.get_rect(center=self.rect.center)
        screen.blit(txt, txt_rect)

# --- NEW COMPONENT: ANGLE WHEEL ---
class AngleWheel:
    
    def __init__(self, x, y, radius=35, initial_angle=0.0):
        self.center = (x, y)
        self.radius = radius
        self.angle = initial_angle
        self.rect = pygame.Rect(x - radius, y - radius, radius * 2, radius * 2)
        self.dragging = False
        
    def handle_event(self, event):
        if event.type == pygame.MOUSEBUTTONDOWN and event.button == 1:
            dist = np.linalg.norm(np.array(event.pos) - np.array(self.center))
            if dist <= self.radius:
                self.dragging = True
                self.update_angle(event.pos)
                return True
                
        elif event.type == pygame.MOUSEBUTTONUP and event.button == 1:
            self.dragging = False
            
        elif event.type == pygame.MOUSEMOTION and self.dragging:
            self.update_angle(event.pos)
            return True
        return False

    def update_angle(self, mouse_pos):
        dx = mouse_pos[0] - self.center[0]
        dy = mouse_pos[1] - self.center[1]
        # Pygame Y is down, so mathematical angle needs adjustment if we want standard trig
        self.angle = np.arctan2(dy, dx)   
        
    def render(self, screen):
        # Update rect position in case center moved (e.g. when form moves)
        self.rect.topleft = (self.center[0] - self.radius, self.center[1] - self.radius)
        
        pygame.draw.circle(screen, (220, 220, 220), self.center, self.radius)
        pygame.draw.circle(screen, (0, 0, 0), self.center, self.radius, 2)
        
        end_x = self.center[0] + self.radius * np.cos(self.angle)
        end_y = self.center[1] + self.radius * np.sin(self.angle)
        pygame.draw.line(screen, (200, 0, 0), self.center, (end_x, end_y), 3)
        pygame.draw.circle(screen, (0, 0, 0), self.center, 4)

class CreationForm:
    def __init__(self):
        self.active = False
        self.rect = pygame.Rect(0, 0, 380, 500) # INCREASED HEIGHT to 500
        self.target_pos = (0,0)
        self.font = pygame.font.SysFont('Arial', 16, bold=True)
        self.edit_mode = False
        self.editing_particle = None
        
        self.creation_memory = {} 

        # --- COMPONENTS ---
        self.btn_remember = BinaryToggle(0, 0, 120, 25, "Remember Input", False)
        self.btn_charge_sign = BinaryToggle(0, 0, 50, 30, "", True, text_true="+", text_false="-", col_true=COL_POS, col_false=COL_NEG)
        self.input_charge_mantissa = MantissaTextBox(0, 0, 80, 30, "1.0")
        self.input_charge_mantissa.allow0 = True
        self.input_charge_exponent = ExponentTextBox(0, 0, 60, 30, "0")
        self.input_mass_mantissa = MantissaTextBox(0, 0, 80, 30, "1.0")
        self.input_mass_exponent = ExponentTextBox(0, 0, 60, 30, "0")
        self.btn_env = BinaryToggle(0, 0, 60, 30, "Env", True)
        self.btn_static = BinaryToggle(0, 0, 60, 30, "Static", False)
        self.slider_restitution = WallCORSlider(0, 0, 160, 20, initial_value=1.0) # change its label
        self.slider_restitution.label = "Restitution:"
        
        # --- VELOCITY COMPONENTS ---
        self.angle_wheel = AngleWheel(0, 0, 35, initial_angle=0.0) # Radius 35
        self.input_vel_mantissa = MantissaTextBox(0, 0, 80, 30, "0")
        self.input_vel_mantissa.allow0 = True
        self.input_vel_exponent = ExponentTextBox(0, 0, 60, 30, "0")
        
        # Buttons
        self.btn_submit = pygame.Rect(0,0, 100, 30)
        self.btn_cancel = pygame.Rect(0,0, 100, 30)
        self.btn_delete = pygame.Rect(0,0, 100, 30)

    def show(self, pos, particle=None):
        self.active = True
        self.target_pos = pos
        self.edit_mode = particle is not None
        self.editing_particle = particle
        self.rect.center = (SW//2, SH//2)
        
        if self.edit_mode and particle:
            charge_val = abs(particle.charge)
            if charge_val == 0: c_mantissa, c_exponent = 0.0, 0
            else:
                import math
                c_exponent = int(math.floor(math.log10(charge_val)))
                c_mantissa = charge_val / (10 ** c_exponent)
            
            mass_val = particle.mass
            if mass_val == 0: m_mantissa, m_exponent = 1.0, 0
            else:
                import math
                m_exponent = int(math.floor(math.log10(mass_val)))
                m_mantissa = mass_val / (10 ** m_exponent)
            
            # Velocity load
            vel_vector = particle.vel_0
            vel_mag = np.linalg.norm(vel_vector)
            vel_angle = np.arctan2(vel_vector[1], vel_vector[0])
            
            if vel_mag == 0: v_mantissa, v_exponent = 0.0, 0
            else:
                import math
                v_exponent = int(math.floor(math.log10(vel_mag)))
                v_mantissa = vel_mag / (10 ** v_exponent)

            self.input_charge_mantissa.text = f"{c_mantissa:.1f}"
            self.input_charge_exponent.text = str(c_exponent)
            self.input_mass_mantissa.text = f"{m_mantissa:.1f}"
            self.input_mass_exponent.text = str(m_exponent)
            
            self.input_vel_mantissa.text = f"{v_mantissa:.1f}"
            self.input_vel_exponent.text = str(v_exponent)
            self.angle_wheel.angle = vel_angle
            
            self.btn_charge_sign.state = particle.charge >= 0
            self.btn_env.state = particle.environmental
            self.btn_static.state = particle.static
            try:
                self.slider_restitution.value = float(particle.e)
            except Exception:
                self.slider_restitution.value = 1.0

        elif self.btn_remember.state and self.creation_memory:
            # LOAD FROM MEMORY
            self.input_charge_mantissa.text = self.creation_memory.get('c_man', "1.0")
            self.input_charge_exponent.text = self.creation_memory.get('c_exp', "0")
            self.input_mass_mantissa.text = self.creation_memory.get('m_man', "1.0")
            self.input_mass_exponent.text = self.creation_memory.get('m_exp', "0")
            self.input_vel_mantissa.text = self.creation_memory.get('v_man', "0.0")
            self.input_vel_exponent.text = self.creation_memory.get('v_exp', "0")
            self.angle_wheel.angle = self.creation_memory.get('v_angle', 0.0)
            
            self.btn_charge_sign.state = self.creation_memory.get('sign', True)
            self.btn_env.state = self.creation_memory.get('env', True)
            self.btn_static.state = self.creation_memory.get('static', True)
            self.slider_restitution.value = self.creation_memory.get('restitution', 1.0)

        else:
            # DEFAULT
            self.input_charge_mantissa.text = "1.0"
            self.input_charge_exponent.text = "0"
            self.input_mass_mantissa.text = "1.0"
            self.input_mass_exponent.text = "0"
            self.input_vel_mantissa.text = "0.0"
            self.input_vel_exponent.text = "0"
            self.angle_wheel.angle = 0.0
            
            self.btn_charge_sign.state = True
            self.btn_env.state = True  
            self.btn_static.state = False
            self.slider_restitution.value = 1.0
            
        # --- LAYOUT CALCULATION (FIXED SPACING) ---
        rx, ry = self.rect.x, self.rect.y
        self.btn_remember.rect.topright = (self.rect.right - 20, ry + 35)
        
        # Charge (Y=100)
        self.btn_charge_sign.rect.topleft = (rx + 40, ry + 100)
        self.input_charge_mantissa.rect.topleft = (rx + 100, ry + 100)
        self.input_charge_exponent.rect.topleft = (rx + 290, ry + 100) # Pushed Right
        
        # Mass (Y=170)
        self.input_mass_mantissa.rect.topleft = (rx + 100, ry + 170)
        self.input_mass_exponent.rect.topleft = (rx + 290, ry + 170) # Pushed Right
        
        # Restitution, need to fix the labels
        self.slider_restitution.rect.topleft = (rx + 100, ry + 240)
        self.slider_restitution.slider_rect.topleft = (rx + 100, ry + 240 + 20 // 2 - 5)
        
        # Velocity (Y=310)
        self.angle_wheel.center = (rx + 75, ry + 300 + 15) # Centered for 35px height box
        self.input_vel_mantissa.rect.topleft = (rx + 140, ry + 300)
        self.input_vel_exponent.rect.topleft = (rx + 270, ry + 300) # Pushed Right

        # Toggles (Y=390)
        self.btn_env.rect.topleft = (rx + 50, ry + 380)
        self.btn_static.rect.topleft = (rx + 200, ry + 380)
        
        # Buttons
        self.btn_submit.bottomleft = (rx + 30, self.rect.bottom - 25)
        self.btn_delete.midbottom = (self.rect.centerx, self.rect.bottom - 25)
        self.btn_cancel.bottomright = (self.rect.right - 30, self.rect.bottom - 25)

    def handle_event(self, event):
        if not self.active: return None

        self.input_charge_mantissa.handle_event(event)
        self.input_charge_exponent.handle_event(event)
        self.input_mass_mantissa.handle_event(event)
        self.input_mass_exponent.handle_event(event)
        self.input_vel_mantissa.handle_event(event)
        self.input_vel_exponent.handle_event(event)
        self.angle_wheel.handle_event(event)
        
        self.btn_remember.handle_event(event)
        self.btn_charge_sign.handle_event(event)
        self.btn_env.handle_event(event)
        self.btn_static.handle_event(event)
        self.slider_restitution.handle_event(event)

        if event.type == pygame.MOUSEBUTTONDOWN and event.button == 1:
            if self.btn_submit.collidepoint(event.pos):
                c_man = self.input_charge_mantissa.get_value()
                c_exp = self.input_charge_exponent.get_value()
                m_man = self.input_mass_mantissa.get_value()
                m_exp = self.input_mass_exponent.get_value()
                v_man = self.input_vel_mantissa.get_value()
                v_exp = self.input_vel_exponent.get_value()
                cor_val = getattr(self.slider_restitution, 'value', 1.0)
                
                # Validation
                if (c_man is not None and c_exp is not None and 
                    m_man is not None and m_exp is not None and
                    v_man is not None and v_exp is not None):
                    
                    if c_man >= 0 and c_man <= 9.9 and 1.0 <= m_man <= 9.9 and v_man >= 0:
                        
                        if self.btn_remember.state and not self.edit_mode:
                            self.creation_memory = {
                                'c_man': self.input_charge_mantissa.text,
                                'c_exp': self.input_charge_exponent.text,
                                'm_man': self.input_mass_mantissa.text,
                                'm_exp': self.input_mass_exponent.text,
                                'v_man': self.input_vel_mantissa.text,
                                'v_exp': self.input_vel_exponent.text,
                                'v_angle': self.angle_wheel.angle,
                                'sign': self.btn_charge_sign.state,
                                'env': self.btn_env.state,
                                'static': self.btn_static.state,
                                'restitution': cor_val
                            }

                        charge = c_man * (10 ** c_exp)
                        if not self.btn_charge_sign.state: charge = -charge
                        mass = m_man * (10 ** m_exp)
                        
                        vel_mag = v_man * (10 ** v_exp)
                        angle = self.angle_wheel.angle
                        vx = vel_mag * np.cos(angle)
                        vy = vel_mag * np.sin(angle)
                        velocity_vector = np.array([vx, vy], dtype=float)
                        
                        if mass > 0:
                            self.active = False
                            result = {
                                "pos": self.target_pos,
                                "charge": charge,
                                "mass": mass,
                                "velocity": velocity_vector,
                                "environmental": self.btn_env.state,
                                "static": self.btn_static.state,
                                "e": np.clip(cor_val, 0.0, 2.0)
                            }
                            
                            if self.edit_mode:
                                result["particle"] = self.editing_particle
                                result["edit"] = True
                            else:
                                result["position"] = np.array(self.target_pos, dtype=float)
                                result["delete"] = False
                            return result
            
            elif self.edit_mode and self.btn_delete.collidepoint(event.pos):
                self.active = False
                return {"delete": True, "particle": self.editing_particle}

            elif self.btn_cancel.collidepoint(event.pos):
                self.active = False
                return None
        return None

    def render(self, screen):
        if not self.active: return
        pygame.draw.rect(screen, COL_BG, self.rect)
        pygame.draw.rect(screen, COL_BORDER, self.rect, 2)
        
        title_text = "Edit Charge" if self.edit_mode else "Create Charge"
        title = self.font.render(title_text, True, COL_TEXT)
        screen.blit(title, (self.rect.x + 20, self.rect.y + 20))
        
        # Labels
        l_chg = self.font.render("Charge:", True, COL_TEXT)
        screen.blit(l_chg, (self.rect.x + 100, self.input_charge_mantissa.rect.top - 20))
        l_mass = self.font.render("Mass:", True, COL_TEXT)
        screen.blit(l_mass, (self.rect.x + 100, self.input_mass_mantissa.rect.top - 20))
        l_cor = self.font.render("Bounce:", True, COL_TEXT)
        screen.blit(l_cor, (self.rect.x + 100, self.slider_restitution.rect.top - 20))
        
        l_vel = self.font.render("Initial Velocity:", True, COL_TEXT)
        screen.blit(l_vel, (self.rect.x + 140, self.input_vel_mantissa.rect.top - 20))

        # Exponent Labels (x 10 ^)
        exp_c = self.font.render("x 10 ^", True, COL_TEXT)
        screen.blit(exp_c, (self.input_charge_mantissa.rect.right + 10, self.input_charge_mantissa.rect.centery - 8))
        exp_m = self.font.render("x 10 ^", True, COL_TEXT)
        screen.blit(exp_m, (self.input_mass_mantissa.rect.right + 10, self.input_mass_mantissa.rect.centery - 8))
        exp_v = self.font.render("x 10 ^", True, COL_TEXT)
        screen.blit(exp_v, (self.input_vel_mantissa.rect.right + 10, self.input_vel_mantissa.rect.centery - 8))

        # Render Components
        self.btn_remember.render(screen)
        self.btn_charge_sign.render(screen)
        self.input_charge_mantissa.render(screen)
        self.input_charge_exponent.render(screen)
        self.input_mass_mantissa.render(screen)
        self.input_mass_exponent.render(screen)
        self.slider_restitution.render(screen)
        self.angle_wheel.render(screen)
        self.input_vel_mantissa.render(screen)
        self.input_vel_exponent.render(screen)
        
        self.btn_env.render(screen)
        self.btn_static.render(screen)
        
        pygame.draw.rect(screen, (100, 200, 100), self.btn_submit)
        pygame.draw.rect(screen, (0,0,0), self.btn_submit, 1)
        submit_text = "CHANGE" if self.edit_mode else "SPAWN"
        ts = self.font.render(submit_text, True, (0,0,0))
        ts_rect = ts.get_rect(center=self.btn_submit.center)
        screen.blit(ts, ts_rect)
        
        pygame.draw.rect(screen, (200, 100, 100), self.btn_cancel)
        pygame.draw.rect(screen, (0,0,0), self.btn_cancel, 1)
        tc = self.font.render("CANCEL", True, (0,0,0))
        tc_rect = tc.get_rect(center=self.btn_cancel.center)
        screen.blit(tc, tc_rect)

        if self.edit_mode:
            pygame.draw.rect(screen, (50, 50, 50), self.btn_delete)
            pygame.draw.rect(screen, (0,0,0), self.btn_delete, 1)
            td = self.font.render("DELETE", True, (255,255,255))
            td_rect = td.get_rect(center=self.btn_delete.center)
            screen.blit(td, td_rect)

# for more sliders

class GenericSlider:
    """A reusable slider for any numeric value."""
    def __init__(self, x, y, width, height, min_val, max_val, initial_val, label, fmt="{:.4f}"):
        self.rect = pygame.Rect(x, y, width, height)
        self.slider_rect = pygame.Rect(x, y + height // 2 - 5, width, 10)
        self.knob_radius = 8
        
        self.min_val = min_val
        self.max_val = max_val
        self.value = initial_val
        self.fmt = fmt # Format string for text (e.g., 4 decimal places)
        self.label = label
        
        self.dragging = False
        self.font = pygame.font.SysFont('Arial', 14)

    def handle_event(self, event):
        if event.type == pygame.MOUSEBUTTONDOWN and event.button == 1:
            # Check knob collision based on current value
            val_norm = (self.value - self.min_val) / (self.max_val - self.min_val)
            knob_x = self.slider_rect.x + val_norm * self.slider_rect.width
            knob_rect = pygame.Rect(knob_x - self.knob_radius, self.slider_rect.y - self.knob_radius, 
                                   self.knob_radius * 2, self.knob_radius * 2)
            
            # Allow clicking anywhere on the bar to jump
            if knob_rect.collidepoint(event.pos) or self.slider_rect.collidepoint(event.pos):
                self.dragging = True
                self.update_from_mouse(event.pos)
                return True
                
        elif event.type == pygame.MOUSEBUTTONUP and event.button == 1:
            self.dragging = False
            
        elif event.type == pygame.MOUSEMOTION and self.dragging:
            self.update_from_mouse(event.pos)
            return True
        return False

    def update_from_mouse(self, mouse_pos):
        relative_x = mouse_pos[0] - self.slider_rect.x
        ratio = max(0.0, min(1.0, relative_x / self.slider_rect.width))
        self.value = self.min_val + ratio * (self.max_val - self.min_val)

    def render(self, screen):
        # Label
        label_surf = self.font.render(self.label, True, (0, 0, 0))
        screen.blit(label_surf, (self.rect.x, self.rect.y - 20))
        
        # Track
        pygame.draw.rect(screen, (200, 200, 200), self.slider_rect)
        pygame.draw.rect(screen, (0, 0, 0), self.slider_rect, 1)
        
        # Normalized position for drawing
        val_norm = (self.value - self.min_val) / (self.max_val - self.min_val)
        
        # Filled portion
        filled_width = self.slider_rect.width * val_norm
        filled_rect = pygame.Rect(self.slider_rect.x, self.slider_rect.y, filled_width, self.slider_rect.height)
        pygame.draw.rect(screen, (100, 150, 255), filled_rect)
        
        # Knob
        knob_x = self.slider_rect.x + val_norm * self.slider_rect.width
        knob_color = (50, 100, 200) if self.dragging else (100, 150, 255)
        pygame.draw.circle(screen, knob_color, (int(knob_x), int(self.slider_rect.centery)), self.knob_radius)
        pygame.draw.circle(screen, (0, 0, 0), (int(knob_x), int(self.slider_rect.centery)), self.knob_radius, 2)
        
        # Value Text
        val_str = self.fmt.format(self.value)
        value_text = self.font.render(val_str, True, (0, 0, 0))
        screen.blit(value_text, (self.slider_rect.right + 10, self.slider_rect.centery - 7))

In [ ]:
%%writefile electric_field_jupyter2.py

# below is electric_field.py file

# necessary imports

import pygame
import numpy as np
from constants_for_all_files import *

# Electric Field Calculation and Visualization

class ElectricFieldSystem:

    def __init__(self):

        # sets up the virtual grid

        self.x_coords = np.linspace(0, SW, V_SW)
        self.y_coords = np.linspace(0, SH, V_SH)
        self.grid_x, self.grid_y = np.meshgrid(self.x_coords, self.y_coords)
        
        # surface to hold the heatmap

        self.surface = pygame.Surface((V_SW, V_SH))

    def render(self, target_surface, charges):

        if not charges:
            target_surface.fill(BGC)
            return

        # Calculate the electric field at each grid point due to all charges

        net_field = np.zeros_like(self.grid_x, dtype=float)

        for pc in charges:
            dx = self.grid_x - pc.position[0]
            dy = self.grid_y - pc.position[1]
            r2 = dx**2 + dy**2
            r2[r2 == 0] = E_CHARGE # Prevent division by zero
            min_distance = pc.total_radius ** 2
            r2[r2 < min_distance] = min_distance # Cap minimum distance squared to avoid extreme values

            contribution = (K_COULOMB * pc.charge) / r2
            net_field += contribution

        # colourises the heatmap based on field strength

        max_val = np.max(np.abs(net_field))
        if max_val == 0:
            max_val = 100  # so eveything is neutral grey

        normalised = 0.7 * net_field / max_val
        pixel_vals = 127.5 - (normalised * 127.5)
        pixel_vals = np.clip(pixel_vals, 0, 255).astype(np.uint8)
        
        # create RGB array for Pygame surface

        pixels = np.dstack([pixel_vals] * 3)
        pygame.surfarray.blit_array(self.surface, np.swapaxes(pixels, 0, 1))
        
        # scale up to full screen

        scaled_view = pygame.transform.scale(self.surface, (SW, SH))
        target_surface.blit(scaled_view, (0, 0))
            

In [ ]:
%%writefile point_charge_jupyter2.py

# below is point_charges.py

# necessary imports

import pygame 
import numpy as np
from constants_for_all_files import *
from collections import deque

# --- ARROW CLASS (ENCAPSULATED) ---
class Arrow:
    def __init__(self, start_pos, length, angle, color=(0, 0, 0), thickness=2):
        self.start_pos = np.array(start_pos, dtype=float)
        self.length = length
        self.angle = angle 
        self.color = color
        self.thickness = thickness

    def update(self, start_pos, length, angle):
        self.start_pos = np.array(start_pos, dtype=float)
        self.length = length
        self.angle = angle

    def render(self, screen):
        if self.length < 1: return 

        # 1. Calculate End Point
        end_x = self.start_pos[0] + self.length * np.cos(self.angle)
        end_y = self.start_pos[1] + self.length * np.sin(self.angle)
        end_pos = (end_x, end_y)

        # 2. Draw Shaft
        pygame.draw.line(screen, self.color, self.start_pos, end_pos, self.thickness)

        # 3. Draw Head
        head_len = 10 
        head_angle = np.radians(150) 
        
        p1_x = end_x + head_len * np.cos(self.angle + head_angle)
        p1_y = end_y + head_len * np.sin(self.angle + head_angle)

        p2_x = end_x + head_len * np.cos(self.angle - head_angle)
        p2_y = end_y + head_len * np.sin(self.angle - head_angle)

        pygame.draw.polygon(screen, self.color, [end_pos, (p1_x, p1_y), (p2_x, p2_y)])


# Point Charge Class

class PointCharge:

    def __init__(self, pos_0, charge, mass, environmental, static, pc_id, e, vel0):
        self.position = np.array(pos_0, dtype=float)
        self.vel = np.array(vel0, dtype=float)
        self.pos_0 = np.array(pos_0, dtype=float)
        self.vel_0 = np.array(vel0, dtype=float)
        self.charge = charge
        self.mass = mass
        self.environmental = environmental
        self.static = static
        self.pc_id = pc_id
        self.dragging = False
        self.e = e

        # Radius setup
        self.core_radius = 15
        self.border_radius = 6
        self.total_radius = self.core_radius + self.border_radius

        if charge > 0:
            self.color = (200, 0, 0)
        elif charge < 0:
            self.color = (0, 0, 200)
        else:
            self.color = (80, 80, 80)  # Darker grey for neutral

        # Trails
        self.history = deque(maxlen=TRAIL_LENGTH)
        self.frame_counter = 0
        self.ghost_surf = pygame.Surface((self.total_radius*2, self.total_radius*2), pygame.SRCALPHA)
        pygame.draw.circle(self.ghost_surf, self.color, (self.total_radius, self.total_radius), int(self.core_radius))

        # --- ARROW SETUP (FIXED) ---
        # Arrow color: black for positive, white for negative
        arrow_color = (0, 0, 0) if charge > 0 else (255, 255, 255)
        self.arrow_obj = Arrow(self.position, 0, 0, color=arrow_color, thickness=3)
        self.arrow_display = True

    @property
    def pos(self):
        """Property to always return current position (for compatibility with electric_field.py)."""
        return self.position
    
    def update_relative_scale(self, all_charges):
        if not all_charges: return

        # --- 1. Radius Calculation (Existing) ---
        # Neutral charges (charge == 0) get middle core radius (12.5px)
        if self.charge == 0:
            self.core_radius = 12.5
        else:
            max_q = max(abs(pc.charge) for pc in all_charges)
            min_q = min(abs(pc.charge) for pc in all_charges)
            q_range = max_q - min_q
            
            current_q = abs(self.charge)
            norm_q = (current_q - min_q) / q_range if q_range != 0 else 0.5
            self.core_radius = 5 + (norm_q * 15)

        # --- 2. Border Calculation (Existing) ---
        max_m = max(pc.mass for pc in all_charges)
        min_m = min(pc.mass for pc in all_charges)
        m_range = max_m - min_m
        
        current_m = self.mass
        norm_m = (current_m - min_m) / m_range if m_range != 0 else 0.5
        self.border_radius = 1 + (norm_m * 4)
        self.total_radius = self.core_radius + self.border_radius
        
        # Update ghost surf
        self.ghost_surf = pygame.Surface((int(self.total_radius*2), int(self.total_radius*2)), pygame.SRCALPHA)
        pygame.draw.circle(self.ghost_surf, self.color, (int(self.total_radius), int(self.total_radius)), int(self.core_radius))

        # --- 3. ARROW UPDATE (New) ---
        # Only update arrows for dynamic particles
        if not self.static:
            # Get magnitude of initial velocity
            my_vel_mag = np.linalg.norm(self.vel_0)
            
            # Find max/min velocity in the system for relative scaling
            velocities = [np.linalg.norm(p.vel_0) for p in all_charges if not p.static]
            if not velocities:
                max_v, min_v = 0, 0
            else:
                max_v, min_v = max(velocities), min(velocities)
            
            # Scale arrow length between 30px and 100px based on relative speed
            v_range = max_v - min_v
            if v_range == 0:
                scale = 0.5 if max_v > 0 else 0
            else:
                scale = (my_vel_mag - min_v) / v_range
            
            length = 0 if my_vel_mag == 0 else 30 + (scale * 70)
            angle = np.arctan2(self.vel_0[1], self.vel_0[0])
            
            # Update the internal Arrow object
            self.arrow_obj.update(self.position, length, angle)

    def reset(self):
        # Always revert to initial state (position and velocity)
        self.position = self.pos_0.copy()
        self.vel = self.vel_0.copy()
        # also reset trail history when resetting particle
        try:
            self.history.clear()
            self.frame_counter = 0
        except Exception:
            pass

    # Helper for angle calc
    @staticmethod
    def calc_phi(v): 
        # returns angle in radians for vector v
        if v[0] == 0 and v[1] == 0: return 0.0
        return np.arctan2(v[1], v[0])

    def update_arrow(self, all_charges):
        """Calculates arrow length and angle based on relative velocity."""
        if not all_charges: return

        if self.charge < 0:
            self.arrow_obj.color = (255, 255, 255)  # White for negative charges
        else:
            self.arrow_obj.color = (0, 0, 0)  # Black for positive charges

        angle = PointCharge.calc_phi(self.vel_0)

        # Scaling logic for length
        # Find max velocity in the system to normalize arrow length
        max_v0 = max(np.linalg.norm(pc.vel_0) for pc in all_charges)
        min_v0 = min(np.linalg.norm(pc.vel_0) for pc in all_charges)
        
        # Avoid zero range div
        if max_v0 == 0:
            length = 0
        else:
            this_v0_mag = np.linalg.norm(self.vel_0)
            # Scale between 20px and 80px
            if max_v0 == min_v0:
                scale = 1.0 if max_v0 > 0 else 0
            else:
                scale = (this_v0_mag - min_v0) / (max_v0 - min_v0)
            
            # Base length 20, max add 60
            length = 20 + (scale * 60) if this_v0_mag > 0 else 0

        self.arrow_obj.update(self.position, length, angle)
    
    def render(self, screen):
        # --- RENDER ARROW FIRST (underneath particle) ---
        # Only if enabled, not static, and velocity is not zero
        if self.arrow_display and not self.static and np.linalg.norm(self.vel_0) > 0:
            self.arrow_obj.render(screen)
        
        # Draw Border
        if self.charge > 0: border_col = (220, 90, 90)
        elif self.charge < 0: border_col = (130, 20, 130)
        else: border_col = (255, 182, 193)  # Light pink for neutral
            
        pygame.draw.circle(screen, border_col, self.position.astype(int), int(self.total_radius))
        
        # Draw Core
        pygame.draw.circle(screen, self.color, self.position.astype(int), int(self.core_radius))
        
        # Selection Highlight
        if self.dragging:
            pygame.draw.circle(screen, (255, 255, 255), self.position.astype(int), int(self.total_radius + 2), 2)
            self.position = np.array(pygame.mouse.get_pos(), dtype=float)

    def record_trail(self, enabled=False):
        """Record current position into history when enabled.
        Records every TRAIL_SKIP frames. Clears history immediately when disabled.
        """
        if enabled:
            self.frame_counter += 1
            if self.frame_counter % TRAIL_SKIP == 0:
                # store a copy to avoid mutation
                self.history.append(self.position.copy())
        else:
            # Clear history immediately when disabled
            if self.history:
                self.history.clear()
            self.frame_counter = 0

    def render_trails(self, screen):
        """Render the stored trail positions with a simple fade effect.
        Older positions are more transparent. Uses small temporary surfaces with per-surface alpha.
        """
        if not self.history:
            return

        length = len(self.history)
        # Draw from oldest to newest so newer points appear on top
        for i, pos in enumerate(self.history):
            if length > 1:
                t = i / (length - 1)
            else:
                t = 1.0
            # Alpha from 40 (oldest) -> 230 (newest)
            alpha = int(40 + (230 - 40) * t)

            # Draw a small filled circle using a temporary surface to allow alpha
            r = max(1, int(self.core_radius))
            surf_size = r * 2 + 4
            surf = pygame.Surface((surf_size, surf_size), pygame.SRCALPHA)
            col = (self.color[0], self.color[1], self.color[2], alpha)
            pygame.draw.circle(surf, col, (surf_size // 2, surf_size // 2), r)
            blit_pos = (int(pos[0] - surf_size // 2), int(pos[1] - surf_size // 2))
            screen.blit(surf, blit_pos)

    def resolve_collision(self, other):
        """
        Resolves elastic collision with another PointCharge.
        Handles position correction (un-sticking) and velocity impulse.
        """
        # 1. Get distance vector and magnitude
        diff = self.position - other.position
        dist_sq = np.dot(diff, diff)
        min_dist = self.total_radius + other.total_radius
        
        # Optimization: Check squared distance first to avoid sqrt() if not needed
        if dist_sq >= min_dist * min_dist:
            return

        # --- COLLISION DETECTED! ---
        dist = np.sqrt(dist_sq)

        # 2. Position Correction (Prevent Sticking/Ghosting)
        # If distance is 0 (exact overlap), pick random direction to avoid divide-by-zero
        if dist == 0:
            n = np.array([1.0, 0.0])
            overlap = min_dist
        else:
            n = diff / dist # Normal vector (Direction of collision)
            overlap = min_dist - dist
        
        # Move particles apart so they just touch
        # If one is static, the other moves the full distance
        if not self.static and not other.static:
            self.position += n * (overlap / 2.0)
            other.position -= n * (overlap / 2.0)
        elif not self.static and other.static:
            self.position += n * overlap
        elif self.static and not other.static:
            other.position -= n * overlap
        else:
            return # Both static, nothing happens

        # 3. Velocity Response (1D Elastic Collision along the Normal)
        # Relative velocity
        v_rel = self.vel - other.vel
        
        # Velocity along the normal
        vel_along_normal = np.dot(v_rel, n)

        # Do not resolve if velocities are separating (they are already moving away)
        if vel_along_normal > 0:
            return

        # Elasticity (1.0 = Perfectly Elastic, 0.0 = Sticky)
        # Use an effective restitution combining both bodies (choose the smaller)
        try:
            e = min(float(self.e), float(other.e))
        except Exception:
            e = 1.0

        # Calculate Impulse Scalar (J)
        # J = -(1 + e) * (v_rel . n) / (1/m1 + 1/m2)
        inv_mass1 = 0 if self.static else 1.0 / self.mass
        inv_mass2 = 0 if other.static else 1.0 / other.mass
        
        # Infinite mass check
        if inv_mass1 + inv_mass2 == 0:
            return

        j = -(1 + e) * vel_along_normal
        j /= (inv_mass1 + inv_mass2)

        # Apply Impulse Vector
        impulse = j * n
        
        if not self.static:
            self.vel += impulse * inv_mass1
        if not other.static:
            other.vel -= impulse * inv_mass2

    def exert_force(self, other):

        # Calculate the electric force exerted on this charge by another charge
        k = 8.9875517873681764e9 # Coulomb's constant in N·m²/C²
        diff = other.position - self.position
        dist = np.linalg.norm(diff)
        if dist == 0:
            return np.array([0.0, 0.0]) # Avoid divide by zero

        force_magnitude = k * abs(self.charge * other.charge) / (dist ** 2)
        force_direction = diff / dist

        # Determine if the force is attractive or repulsive
        if self.charge * other.charge > 0:
            force_direction = -force_direction # Repulsive force

        force = force_direction * force_magnitude
        return force

In [ ]:
%%writefile physics_engine_jupyter2.py

# physics_engine.py
import numpy as np

# Keep your existing constants import in your file
from constants_for_all_files import *  # K_COULOMB, WALL_INNER_RECT, etc.

class PhysicsEngine:
    """
    Symplectic/Velocity-Verlet based physics engine for point charges with:
    - softening to avoid singularities
    - pairwise particle collision resolution (positional correction + impulse)
    - wall collisions
    - energy diagnostics (kinetic + Coulomb potential)
    """

    def __init__(self, softening_eps=100.0, min_r2=400.0, debug=False):
        """
        softening_eps: length scale used to soften Coulomb potential (pixels)
        min_r2: fallback minimum r^2 in get_accelerations (keeps your previous safety)
        debug: if True, print energy diagnostics occasionally
        """
        self.softening_eps = float(softening_eps)
        self.softening_eps2 = self.softening_eps * self.softening_eps
        self.min_r2 = float(min_r2)
        self.debug = debug

    # ----- Force / Acceleration -----
    def get_accelerations(self, positions, velocities, charges, masses, static_status):
        """
        positions: (N,2) array
        velocities: (N,2) array (not used directly for Coulomb force, but kept for API)
        charges: (N,) array
        masses: (N,) array
        static_status: (N,) boolean array, True if static (immovable)
        returns: accelerations array shape (N,2)
        """
        n = len(positions)
        accelerations = np.zeros((n, 2), dtype=float)
        if n < 2:
            return accelerations

        # Pairwise O(N^2) Coulomb. Use softening by adding eps^2 to r^2.
        for i in range(n):
            if static_status[i]:
                continue
            net_force = np.zeros(2, dtype=float)
            pos_i = positions[i]
            qi = charges[i]

            for j in range(n):
                if i == j:
                    continue

                pos_j = positions[j]
                qj = charges[j]
                diff = pos_i - pos_j      # r_i - r_j (force on i)
                r2 = np.dot(diff, diff)

                # Respect minimum distance for stability (legacy fallback)
                if r2 < self.min_r2:
                    r2 = self.min_r2

                # Softening addition for Coulomb denominator
                r2_soft = r2 + self.softening_eps2
                r = np.sqrt(r2_soft)

                # Coulomb force magnitude: k * qi * qj / r^2_soft
                # direction: diff / r
                if r > 0:
                    force_mag = (K_COULOMB * qi * qj) / r2_soft
                    direction = diff / r
                    force = force_mag * direction
                    net_force += force
                # else r extremely small, skip (shouldn't happen due to softening)

            accelerations[i] = net_force / masses[i]

        return accelerations

    # ----- Symplectic Integrator: Velocity-Verlet -----
    def update_positions_velocities(self, dt, all_charges):
        """
        Velocity-Verlet update for all particles.
        Steps:
            1) compute a(t)
            2) x(t+dt) = x(t) + v(t)*dt + 0.5*a(t)*dt^2
            3) compute a(t+dt) from new positions
            4) v(t+dt) = v(t) + 0.5*(a(t) + a(t+dt))*dt
        """
        n = len(all_charges)
        if n == 0:
            return

        positions = np.array([p.position for p in all_charges])
        velocities = np.array([p.vel for p in all_charges])
        charges = np.array([p.charge for p in all_charges])
        masses = np.array([p.mass for p in all_charges])
        static_status = np.array([p.static for p in all_charges], dtype=bool)

        # 1. initial accelerations a(t)
        a_t = self.get_accelerations(positions, velocities, charges, masses, static_status)

        # 2. update positions
        positions_new = positions + velocities * dt + 0.5 * a_t * (dt * dt)

        # Optional: keep static particles exactly fixed
        for i, s in enumerate(static_status):
            if s:
                positions_new[i] = positions[i]

        # 3. compute accelerations at new positions a(t+dt)
        # Note: velocities passed here are still v(t) — that's fine for force calc
        a_tdt = self.get_accelerations(positions_new, velocities, charges, masses, static_status)

        # 4. update velocities
        velocities_new = velocities + 0.5 * (a_t + a_tdt) * dt

        # 5. write back into objects (skip static)
        for i in range(n):
            if not static_status[i]:
                all_charges[i].position = positions_new[i]
                all_charges[i].vel = velocities_new[i]

    # ----- Pairwise particle collision resolution (internal) -----
    def _resolve_pair_collision(self, p1, p2):
        """
        Robust collision handling between two point-charge objects.
        - Impulse resolution using effective restitution = min(e1, e2)
        - Positional correction (push them apart based on overlap)
        - Handles static flags and infinite mass
        Assumes p1 and p2 expose:
            position (np.array([x,y])), vel (np.array([vx,vy])),
            total_radius (hitbox), mass, static (bool), e (per-object)
        """
        # Quick exit if both static
        if p1.static and p2.static:
            return

        diff = p1.position - p2.position
        dist_sq = np.dot(diff, diff)
        min_dist = p1.total_radius + p2.total_radius

        # If not overlapping, nothing to do
        if dist_sq >= (min_dist * min_dist):
            return

        # compute distance and normal
        if dist_sq == 0:
            # perfect overlap - choose arbitrary normal
            n = np.array([1.0, 0.0], dtype=float)
            dist = 0.0
            overlap = min_dist
        else:
            dist = np.sqrt(dist_sq)
            n = diff / dist
            overlap = min_dist - dist

        # -------- impulse (velocity) resolution FIRST --------
        inv_m1 = 0.0 if p1.static else (1.0 / p1.mass)
        inv_m2 = 0.0 if p2.static else (1.0 / p2.mass)
        inv_mass_sum = inv_m1 + inv_m2

        if inv_mass_sum == 0:
            return  # both infinite mass

        # relative velocity along normal
        v_rel = p1.vel - p2.vel
        vel_along_normal = np.dot(v_rel, n)

        # If separating (positive), nothing to do
        if vel_along_normal > 0:
            return

        # effective restitution: choose minimum (less bouncy dominates)
        e_eff = min(getattr(p1, "e", 1.0), getattr(p2, "e", 1.0))

        # compute impulse scalar
        j = -(1.0 + e_eff) * vel_along_normal
        j /= inv_mass_sum

        impulse = j * n

        if not p1.static:
            p1.vel += impulse * inv_m1
        if not p2.static:
            p2.vel -= impulse * inv_m2

        # -------- positional correction AFTER (with reduced correction to minimize energy injection) --------
        # Use only 80% correction to avoid overshooting and energy gain
        correction_factor = 0.8
        
        if not p1.static and not p2.static:
            p1.position += n * (overlap * correction_factor * (inv_m1 / inv_mass_sum))
            p2.position -= n * (overlap * correction_factor * (inv_m2 / inv_mass_sum))
        elif not p1.static and p2.static:
            p1.position += n * (overlap * correction_factor)
        elif p1.static and not p2.static:
            p2.position -= n * (overlap * correction_factor)

    # ----- Public collision handlers -----
    def handle_particle_collisions(self, all_charges):
        """O(N^2) pairwise collision checks and resolves using internal resolver."""
        n = len(all_charges)
        for i in range(n):
            for j in range(i + 1, n):
                p1 = all_charges[i]
                p2 = all_charges[j]
                # quick skip if both static
                if p1.static and p2.static:
                    continue
                self._resolve_pair_collision(p1, p2)

    def handle_wall_collisions(self, all_charges, wall_cor):
        """
        Boundary collisions — inverts velocity component and multiplies by wall_cor (coefficient).
        Keeps particles inside WALL_INNER_RECT (assumes pygame Rect-like object).
        """
        for pc in all_charges:
            if pc.static:
                continue

            left_bound = WALL_INNER_RECT.left + pc.total_radius
            right_bound = WALL_INNER_RECT.right - pc.total_radius
            if pc.position[0] < left_bound:
                pc.position[0] = left_bound
                pc.vel[0] *= -wall_cor
            elif pc.position[0] > right_bound:
                pc.position[0] = right_bound
                pc.vel[0] *= -wall_cor

            top_bound = WALL_INNER_RECT.top + pc.total_radius
            bottom_bound = WALL_INNER_RECT.bottom - pc.total_radius
            if pc.position[1] < top_bound:
                pc.position[1] = top_bound
                pc.vel[1] *= -wall_cor
            elif pc.position[1] > bottom_bound:
                pc.position[1] = bottom_bound
                pc.vel[1] *= -wall_cor

    # ----- Energy diagnostics -----
    def compute_energy(self, all_charges):
        """
        Returns (kinetic_energy, potential_energy) as floats.
        Potential energy is Coulomb pairwise sum: U = sum_{i<j} k qi qj / r_soft
        (Note: r_soft uses softening to avoid singularity.)
        """
        n = len(all_charges)
        positions = np.array([p.position for p in all_charges])
        velocities = np.array([p.vel for p in all_charges])
        charges = np.array([p.charge for p in all_charges])
        masses = np.array([p.mass for p in all_charges])
        static_status = np.array([p.static for p in all_charges], dtype=bool)

        # kinetic energy
        ke = 0.0
        for i in range(n):
            v2 = np.dot(velocities[i], velocities[i])
            ke += 0.5 * masses[i] * v2

        # potential energy (pairwise Coulomb)
        pe = 0.0
        for i in range(n):
            for j in range(i + 1, n):
                diff = positions[i] - positions[j]
                r2 = np.dot(diff, diff)
                # use min fallback and softening consistent with forces
                if r2 < self.min_r2:
                    r2 = self.min_r2
                r = np.sqrt(r2 + self.softening_eps2)
                pe += K_COULOMB * charges[i] * charges[j] / r

        return ke, pe

    # Helper to step full physics tick: integrate, particle collisions, wall collisions, optional energy print
    def step(self, dt, all_charges, wall_cor=1.0, do_collisions=True, do_walls=True, diagnostics=False):
        """
        Convenience function performing:
            - integrate (Velocity-Verlet)
            - handle particle collisions (if do_collisions)
            - handle wall collisions (if do_walls)
            - return energy diagnostics if diagnostics True
        """
        # 1) Integrate
        self.update_positions_velocities(dt, all_charges)

        # 2) Resolve particle collisions (positional + impulses)
        if do_collisions:
            self.handle_particle_collisions(all_charges)

        # 3) Wall collisions
        if do_walls:
            self.handle_wall_collisions(all_charges, wall_cor)

        # 4) Diagnostics
        if diagnostics or self.debug:
            ke, pe = self.compute_energy(all_charges)
            if self.debug:
                print(f"[PhysicsEngine] KE: {ke:.6f}  PE: {pe:.6f}  Total: {ke+pe:.6f}")
            return ke, pe

        return None


In [ ]:
%%writefile main_jupyter2.py

import pygame
import sys
import numpy as np

# IMPORTS
from constants_for_all_files_jupyter2 import *
from point_charge_jupyter2 import *
from electric_field_jupyter2 import *
from physics_engine_jupyter2 import *
from gui_jupyter2 import *

# --- 1. INITIALIZATION & SETUP ---
pygame.init()
screen = pygame.display.set_mode((SW, SH))
pygame.display.set_caption("FREE1105: Phase 2")
clock = pygame.time.Clock()

# --- 2. SYSTEMS ---
ef_system = ElectricFieldSystem()
reset_btn = ResetButton()
pause_btn = PauseButton()
unpause_btn = UnpauseButton()
start_btn = StartButton()
context_menu = ContextMenu()
create_form = CreationForm()
trails_toggle = TrailsToggle()

# Sliders
wall_cor_slider = WallCORSlider(30, 20, 150, 30, initial_value=BW_coeff)
dt_slider = GenericSlider(1250, 50, 150, 30,min_val=0.00005, max_val=0.02, initial_val=1/1000, 
                          label="Time Step (dt):", fmt="{:.5f}")
fps_slider = GenericSlider(1250, 100, 150, 30, min_val=60, max_val=9000, initial_val=100, 
                           label="Max FPS:", fmt="{:.0f}")

# engine
physics_engine = PhysicsEngine()

# --- 3. STATE MANAGEMENT ---
# 0 = SETUP (Edit Mode), 1 = RUNNING (Physics Mode)
sim_state = 0 

all_point_charges = []
selected_charge = None
p_id_counter = 0
wall_cor = wall_cor_slider.value 
# Double-click tracking for particle duplication
LAST_CLICK_MS = 400  # max ms between clicks to consider a double-click
last_click_time = 0
last_click_pc_id = None

# Energy tracking
total_kinetic_energy = 0.0
total_potential_energy = 0.0
initial_total_energy = None  # Set when simulation starts
energy_font = pygame.font.SysFont('Arial', 16)

# Pause text bouncing (DVD logo style)

import math
pause_text_pos = [SW // 2, SH // 2]
pause_text_vel = [900 * math.cos(math.radians(40)), 900 * math.sin(math.radians(40))]  # 40 degrees
pause_font = pygame.font.SysFont("Roboto", 74)

# core simulation loop

running = True
while running:
    
    dt = dt_slider.value
    fps = int(fps_slider.value)

    # --- A. TIME & MOUSE ---
    mouse_pos = pygame.mouse.get_pos()
    
    # --- B. GLOBAL EVENT HANDLING (THE ONE TRUE LOOP) ---
    for event in pygame.event.get():
        # 1. GLOBAL: Quit
        if event.type == pygame.QUIT:
            running = False
        
        # 2. GLOBAL: Reset Button (Works in ANY state to save you)
        if reset_btn.handle_event(event):
            sim_state = 0 # Force back to SETUP
            # Reset all charges to initial positions and velocities
            for pc in all_point_charges:
                pc.reset()
            print("SIMULATION RESET")

        # --- C. STATE SPECIFIC EVENTS ---
        
        # === STATE 0: SETUP PHASE (Editing Allowed) ===
        if sim_state == 0:
            
            # 1. Start Button (Only works in Setup) and requires no particles touching walls or other particles and they are existing
            particles_touching = False
            for pc in all_point_charges:
                # Check against the visible black wall (WALL_INNER_RECT), not the window edges
                if (pc.position[0] - pc.total_radius <= WALL_INNER_RECT.left or
                    pc.position[0] + pc.total_radius >= WALL_INNER_RECT.right or
                    pc.position[1] - pc.total_radius <= WALL_INNER_RECT.top or
                    pc.position[1] + pc.total_radius >= WALL_INNER_RECT.bottom):
                    particles_touching = True
                    break
                for other_pc in all_point_charges:
                    if other_pc != pc:
                        dist = np.linalg.norm(pc.position - other_pc.position)
                        if dist < (pc.total_radius + other_pc.total_radius):
                            particles_touching = True
                            break
                if particles_touching:
                    break

            # Evaluate start button once per event to avoid duplicate calls
            start_pressed = start_btn.handle_event(event)
            if start_pressed:
                if particles_touching:
                    print("ERROR: Cannot start simulation while particles are touching walls or each other.")
                elif len(all_point_charges) == 0:
                    print("ERROR: Cannot start simulation without any point charges present.")
                else:
                    sim_state = 1 # LOCK IN! Switch to Running
                    # Clear any active selections
                    selected_charge = None
                    context_menu.active = False
                    create_form.active = False
                    # Capture initial energy
                    ke_init, pe_init = physics_engine.compute_energy(all_point_charges)
                    initial_total_energy = ke_init + pe_init
                    print("SIMULATION STARTED")
                    print(f"Initial Total Energy: {initial_total_energy:.2e} J")

            # 2. Sliders
            wall_cor_slider.handle_event(event)
            dt_slider.handle_event(event)
            fps_slider.handle_event(event)

            # 3. Context Menu Logic
            menu_action = context_menu.handle_event(event)
            if menu_action == "open_creator":
                create_form.show(context_menu.pos)

            # 4. Creation Form Logic
            form_data = create_form.handle_event(event)
            if form_data:
                try:
                    # Deletion requested from edit form
                    if form_data.get('delete'):
                        particle = form_data.get('particle')
                        try:
                            all_point_charges = [pc for pc in all_point_charges if pc != particle]
                        except Exception:
                            pass
                        print("Particle deleted")
                        continue
                    if form_data.get('edit'):
                        # Edit existing particle
                        particle = form_data['particle']
                        particle.charge = form_data['charge']
                        particle.mass = form_data['mass']
                        # update coefficient of restitution if provided
                        particle.e = form_data.get('e', getattr(particle, 'e', 1.0))
                        particle.environmental = form_data['environmental']
                        particle.static = form_data['static']
                        # Update color based on charge (handle neutral)
                        if particle.charge > 0:
                            particle.color = (200, 0, 0)
                        elif particle.charge < 0:
                            particle.color = (0, 0, 200)
                        else:
                            particle.color = (80, 80, 80)
                        particle.update_relative_scale(all_point_charges)
                        # update velocity if provided <--- FIX IS HERE
                        particle.vel_0 = form_data['velocity'] 
                        particle.vel = form_data['velocity'].copy()
                    else:
                        # Create new particle
                        new_pc = PointCharge(
                            form_data['position'],
                            form_data['charge'],
                            form_data['mass'],
                            form_data['environmental'],
                            form_data['static'],
                            p_id_counter,
                            form_data.get('e', 1.0),
                            form_data.get('velocity', np.array([0.0, 0.0])) # NEW: Pass vector
                        )
                        all_point_charges.append(new_pc)
                        p_id_counter += 1
                except Exception as e:
                    print(f"Form Error: {e}")

            # 5. Mouse Interactions (Drag & Drop / Right Click)
            # Only allowed if GUI is NOT blocking the mouse
            if event.type == pygame.MOUSEBUTTONDOWN:
                if not context_menu.active and not create_form.active and not wall_cor_slider.dragging:
                    
                    if event.button == 1: # Left Click (Select/Drag / Double-Click Duplicate)
                        clicked_pc = None
                        click_pos = np.array(event.pos)
                        for pc in reversed(all_point_charges):
                            dist = np.linalg.norm(pc.position - click_pos)
                            if dist < pc.total_radius:
                                clicked_pc = pc
                                break

                        if clicked_pc is not None:
                            now = pygame.time.get_ticks()
                            # Double-click detection: same particle clicked twice within LAST_CLICK_MS
                            if last_click_pc_id == getattr(clicked_pc, 'pc_id', None) and (now - last_click_time) <= LAST_CLICK_MS:
                                # Duplicate particle slightly shifted to the right and down
                                try:
                                    shift = np.array([12.0, 12.0])
                                    new_pos = clicked_pc.position + shift
                                    new_pc = PointCharge(
                                        new_pos.copy(),
                                        clicked_pc.charge,
                                        clicked_pc.mass,
                                        clicked_pc.environmental,
                                        clicked_pc.static,
                                        p_id_counter,
                                        getattr(clicked_pc, 'e', 1.0)
                                        ,clicked_pc.vel_0
                                    )
                                    all_point_charges.append(new_pc)
                                    p_id_counter += 1
                                    print(f"Duplicated particle {getattr(clicked_pc,'pc_id','?')} -> {p_id_counter-1}")
                                except Exception as e:
                                    print(f"Duplication error: {e}")
                                # reset click tracker to avoid triple-trigger
                                last_click_time = 0
                                last_click_pc_id = None
                            else:
                                # Single click: prepare for drag
                                selected_charge = clicked_pc
                                clicked_pc.dragging = True
                                # store click info for possible double-click
                                last_click_time = pygame.time.get_ticks()
                                last_click_pc_id = getattr(clicked_pc, 'pc_id', None)
                                
                    elif event.button == 3: # Right Click (Context Menu)
                        clicked_on_charge = False
                        clicked_charge = None
                        for pc in reversed(all_point_charges):
                             dist = np.linalg.norm(pc.position - np.array(event.pos))
                             if dist < pc.total_radius:
                                 clicked_on_charge = True
                                 clicked_charge = pc
                                 break
                        
                        if clicked_on_charge:
                            create_form.show(event.pos, particle=clicked_charge)
                        else:
                            context_menu.show(event.pos)

            elif event.type == pygame.MOUSEBUTTONUP:
                if event.button == 1 and selected_charge:
                    selected_charge.dragging = False
                    # Set new initial position to current position after drag
                    selected_charge.pos_0 = selected_charge.position.copy()
                    selected_charge = None
                    
            elif event.type == pygame.MOUSEMOTION:
                if selected_charge:
                    selected_charge.position = np.array(event.pos, dtype=float)

        # === STATE 1: RUNNING PHASE (Physics Allowed) ===
        if sim_state == 1:
            # No dragging allowed here!
            # Just listening for "Pause" (Future Phase) or "Reset" (Global) and fps / dt sliders

            wall_cor_slider.handle_event(event)
            dt_slider.handle_event(event)
            fps_slider.handle_event(event)

            if pause_btn.handle_event(event):
                print(f"DEBUG: Pause clicked at {getattr(event, 'pos', None)}, create_form.active={create_form.active}, context_menu.active={context_menu.active}")
                sim_state = 0.5 # Literally just pauses the sim

            elif reset_btn.handle_event(event):
                sim_state = 0

                # keep only environmental charges and reverts everything to their initial state

                all_point_charges = [pc for pc in all_point_charges if pc.environmental]
                for pc in all_point_charges:
                    pc.reset()

            # Trails toggle (only active in running state)
            if trails_toggle.handle_event(event):
                # handle_event already toggles internal state; when turned off, clear existing trails
                if not trails_toggle.state:
                    for pc in all_point_charges:
                        try:
                            pc.history.clear()
                            pc.frame_counter = 0
                        except Exception:
                            pass

        elif sim_state == 0.5:

            # Paused State - Listen for Unpause from same button or Reset
            if unpause_btn.handle_event(event):
                print(f"DEBUG: Unpause clicked at {getattr(event, 'pos', None)}, create_form.active={create_form.active}, context_menu.active={context_menu.active}")
                sim_state = 1 # Resume Simulation

            elif reset_btn.handle_event(event):
                sim_state = 0

                # keep only environmental charges and reverts everything to their initial state

                all_point_charges = [pc for pc in all_point_charges if pc.environmental]
                for pc in all_point_charges:
                    pc.reset()

    # --- D. LOGIC UPDATES (General) ---
    
    # Update wall coefficient
    wall_cor = wall_cor_slider.value 

    # Update visual scales (Always needed for rendering)
    for pc in all_point_charges:
        pc.update_relative_scale(all_point_charges)

    # --- E. PHYSICS UPDATES (State Dependent) ---

    if sim_state == 1:

        # Get arrays for physics engine
        positions = np.array([p.pos for p in all_point_charges])
        velocities = np.array([p.vel for p in all_point_charges])
        charges = np.array([p.charge for p in all_point_charges])
        masses = np.array([p.mass for p in all_point_charges])
        static_status = np.array([p.static for p in all_point_charges], dtype=bool)

        # 1. The Velocity-Verlet integration step (Motion with forces)
        physics_engine.get_accelerations(positions, velocities, charges, masses, static_status)
        physics_engine.update_positions_velocities(dt, all_point_charges)
        
        # 2. Handle collisions AFTER integration
        physics_engine.handle_wall_collisions(all_point_charges, wall_cor)
        physics_engine.handle_particle_collisions(all_point_charges)
        
        # 3. Energy correction (only if all collisions are perfectly elastic)
        # Check if all particles have e=1.0 and wall_cor=1.0
        all_elastic = all(getattr(pc, 'e', 1.0) == 1.0 for pc in all_point_charges) and wall_cor == 1.0
        
        if all_elastic and initial_total_energy is not None:
            # Calculate current energy
            current_ke, current_pe = physics_engine.compute_energy(all_point_charges)
            current_total = current_ke + current_pe
            
            # Only correct if there's actually kinetic energy to scale
            if current_ke > 1e-10 and abs(current_total - initial_total_energy) > 1e-6:
                # Scale velocities to restore energy
                scale_factor = np.sqrt(max(0, (initial_total_energy - current_pe) / current_ke))
                for pc in all_point_charges:
                    if not pc.static:
                        pc.vel *= scale_factor

        # Record trails for each particle (only if toggle enabled)
        for pc in all_point_charges:
            pc.record_trail(trails_toggle.state)
        
        # Calculate energies using physics engine (ensures consistency with force calculations)
        total_kinetic_energy, total_potential_energy = physics_engine.compute_energy(all_point_charges)

    # Update pause text position (bouncing DVD logo style)
    elif sim_state == 0.5:
        pause_text_pos[0] += pause_text_vel[0] * dt*10
        pause_text_pos[1] += pause_text_vel[1] * dt*19
        
        # Bounce off walls (with margin for text width/height)
        text_bounds = pause_font.render("PAUSED", True, (0, 0, 0)).get_rect()
        text_width = text_bounds.width
        text_height = text_bounds.height
        
        if pause_text_pos[0] - text_width//2 <= 0 or pause_text_pos[0] + text_width//2 >= SW:
            pause_text_vel[0] *= -1
            pause_text_pos[0] = max(text_width//2, min(SW - text_width//2, pause_text_pos[0]))
        
        if pause_text_pos[1] - text_height//2 <= 0 or pause_text_pos[1] + text_height//2 >= SH:
            pause_text_vel[1] *= -1
            pause_text_pos[1] = max(text_height//2, min(SH - text_height//2, pause_text_pos[1]))

    # --- F. RENDERING (The Layers) ---
    
    # Layer 0: Background (E-Field)
    ef_system.render(screen, all_point_charges)
    
    # Layer 1: Environment (Walls)
    pygame.draw.rect(screen, WC, WALL_RECT, WBT) 
    
    # Layer 2: Particles (render trails behind particles)
    
    for pc in all_point_charges:
        # Only show arrows in setup mode (sim_state == 0)
        if sim_state == 0:
            pc.update_arrow(all_point_charges)
            pc.arrow_display = True
        else:
            pc.arrow_display = False
        pc.render_trails(screen)
        pc.render(screen)

    # Layer 3: GUI (State Dependent)

    # Always show Reset
    reset_btn.render(screen)
    
    if sim_state == 0:
        # Only show Setup GUI in Setup Mode
        start_btn.render(screen)
        wall_cor_slider.render(screen)
        context_menu.render(screen)
        create_form.render(screen)
        dt_slider.render(screen)
        fps_slider.render(screen)

    elif sim_state == 1:
        pause_btn.render(screen)
        trails_toggle.render(screen)
        dt_slider.render(screen)
        fps_slider.render(screen)

    elif sim_state == 0.5:
        unpause_btn.render(screen)
        
        # Render PAUSED text with bouncing and letter borders
        pause_text = "PAUSED"
        letter_color = (250, 190, 200)  # Pink
        border_color = (60, 200, 60)  # Light Green
        border_width = 3
        
        # Render each letter individually with border
        letter_spacing = 10
        total_width = sum(pause_font.render(char, True, letter_color).get_width() for char in pause_text) + (len(pause_text) - 1) * letter_spacing
        start_x = pause_text_pos[0] - total_width // 2
        
        for i, char in enumerate(pause_text):
            char_surf = pause_font.render(char, True, letter_color)
            char_width = char_surf.get_width()
            
            # Draw border around letter (by rendering text 4 times around it)
            for dx, dy in [(-border_width, 0), (border_width, 0), (0, -border_width), (0, border_width)]:
                border_surf = pause_font.render(char, True, border_color)
                screen.blit(border_surf, (start_x + dx, pause_text_pos[1] - char_surf.get_height()//2 + dy))
            
            # Draw actual letter on top
            screen.blit(char_surf, (start_x, pause_text_pos[1] - char_surf.get_height()//2))
            start_x += char_width + letter_spacing

    pygame.display.flip()
    clock.tick(fps)

    for pc in all_point_charges:
        print(f"ID {getattr(pc, 'pc_id', 'N/A')}: Pos {pc.pos}, Vel {pc.vel}, Pos_0 {pc.pos_0}, Vel_0 {pc.vel_0}")

pygame.quit()
sys.exit()

In [ ]:
from main_jupyter2 import *

## Phase 3 ##

Core Objectives:
- Implement rigid bodies
- Rigid bodies have presets: triangles (editable via hypotentuse length and angle of adjescant), rectangles (edit x length and y length)
- Rigid Bodies can be dragged and rotated and scale by user up to like 40px radius and down to 10 px radius
- Rigid Bodies can then be charged and given charge distributions, via a interactable wavefunction to dsitribute the charge across the x or y axis of the object
- Add Strings which can be placed between any particle on the screen.

Rough Outline of Implementation:

1) Gonna have to make a new file and class system for rigid bodies